# BBUR burned-area Forecaster (lead-1) - Mainland


Install the required packages.


In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['catboost', 'xgboost', 'lightgbm', 'cftime', 'scikit-learn']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        pip_install(pkg)

print('Dependencies OK')


Mirror stdout/stderr to a timestamped log file.


In [ ]:
import sys, os, datetime, atexit

_LOG_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
_LOG_TS  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_PATH = os.path.join(_LOG_DIR, f'bbur_run_log_{_LOG_TS}.txt')

class _Tee:
    def __init__(self, stream, log_file):
        self._stream = stream
        self._log_file = log_file
    def write(self, data):
        self._stream.write(data)
        self._log_file.write(data)
        self._log_file.flush()
    def flush(self):
        self._stream.flush()
        self._log_file.flush()
    def isatty(self):
        return False

_log_fh = open(LOG_PATH, 'a', buffering=1)
_log_fh.write(f"\\n{'='*80}\n RUN STARTED {datetime.datetime.now().isoformat()}\n{'='*80}\n")
sys.stdout = _Tee(sys.stdout, _log_fh)
sys.stderr = _Tee(sys.stderr, _log_fh)

def _close_log():
    try:
        _log_fh.write(f"\\n{'='*80}\n RUN ENDED {datetime.datetime.now().isoformat()}\n{'='*80}\n")
        _log_fh.flush(); _log_fh.close()
    except Exception:
        pass

atexit.register(_close_log)
print(f"[Logging] All stdout/stderr from this point on are mirrored to: {LOG_PATH}")


Load the training dataset, GFED4s burned-area targets and optional GRACE data.


In [ ]:
import os, gc, subprocess
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

try:
    GPU_AVAILABLE = subprocess.run(['nvidia-smi'], capture_output=True, text=True).returncode == 0
except FileNotFoundError:
    GPU_AVAILABLE = False
print(f'GPU available: {GPU_AVAILABLE}' + (' -- CatBoost/XGBoost (tuning) will use CUDA; per-pixel loop stays CPU regardless' if GPU_AVAILABLE else ' -- running on CPU'))

DATASET_PATH = '/kaggle/input/datasets/mahadiqbal061/bbur-enhanced/bbur_enhanced_training_ready_seasia_025.npz'
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('..', 'data', 'BBUR_enhanced_kaggle_025',
                                'bbur_enhanced_training_ready_seasia_025.npz')

arr           = np.load(DATASET_PATH, allow_pickle=True)
x_flat        = arr['x_flat'].astype(np.float32, copy=False)
y_flat        = arr['y_flat'].astype(np.float32, copy=False)
time_idx      = arr['time_idx'].astype(np.int32, copy=False)
lat_flat      = arr['lat_flat'].astype(np.float32, copy=False)
lon_flat      = arr['lon_flat'].astype(np.float32, copy=False) if 'lon_flat' in arr else None
feature_names = arr['feature_names'].astype(str).tolist()
years_pm      = arr['years_pm'].astype(np.int32, copy=False)
lat           = arr['lat'].astype(np.float32, copy=False)
lon           = arr['lon'].astype(np.float32, copy=False)
del arr; gc.collect()

print(f'Grid   : {len(lat)} lat x {len(lon)} lon = {len(lat)*len(lon):,} pixels  (0.25deg)')
print(f'x_flat : {x_flat.shape}  ({x_flat.nbytes/1e9:.2f} GB in memory)')

# load gfed4s target
GFED_TARGET_PATH = '/kaggle/input/datasets/mahadiqbal061/bbur-enhanced/gfed4s_targets_seasia_025.npz'
if not os.path.exists(GFED_TARGET_PATH):
    GFED_TARGET_PATH = '/kaggle/input/datasets/mahadiqbal061/bbur-enhanced/gfed4s_targets_seasia_025.npz'
if os.path.exists(GFED_TARGET_PATH):
    print("Loading TRUE GFED targets from", GFED_TARGET_PATH)
    gfed_arr  = np.load(GFED_TARGET_PATH)
    y_true_ba = gfed_arr['y_total_flat'].astype(np.float32, copy=False)
    ba_idx    = next(i for i, f in enumerate(feature_names) if 'burned_area' in f.lower())
    x_flat[:, ba_idx] = y_true_ba
    print('Successfully overwrote burned_area_pred with GFED target.')

    del gfed_arr, y_true_ba; gc.collect()

    # lead-time target shift
    LEAD_TIME = 1
    print(f"\\nApplying {LEAD_TIME}-month lead time shift to targets...")
    N_PIXELS = len(lat) * len(lon)
    N_MONTHS = len(years_pm)

    ba_2d = x_flat[:, ba_idx].reshape(N_MONTHS, N_PIXELS)
    shifted_ba = np.zeros_like(ba_2d)
    shifted_ba[:-LEAD_TIME, :] = ba_2d[LEAD_TIME:, :]
    x_flat[:, ba_idx] = shifted_ba.reshape(-1)

    flux_2d = y_flat.reshape(N_MONTHS, N_PIXELS)
    shifted_flux = np.zeros_like(flux_2d)
    shifted_flux[:-LEAD_TIME, :] = flux_2d[LEAD_TIME:, :]
    y_flat = shifted_flux.reshape(-1)

    print('Target arrays shifted successfully.')

else:
    print('WARNING: GFED targets not found! Using default BBUR targets.')

if lon_flat is None:
    raise RuntimeError('lon_flat missing from .npz')

# merge grace data
import xarray as xr
if ' _BASE' not in dir() and '_BASE' not in dir():
    _BASE = os.path.dirname(DATASET_PATH)
GRACE_PATH = os.path.join(_BASE, 'GRCTellus.JPL.200204_202603.GLO.RL06.3M.MSCNv04CRI.nc')
if os.path.exists(GRACE_PATH):
    print('Loading GRACE TWS data (ffill/bfill gap-filling, pre-test 2003-2011 standardization)...')
    ds_grace   = xr.open_dataset(GRACE_PATH)
    lwe        = ds_grace['lwe_thickness']
    scale      = ds_grace['scale_factor']
    tws_scaled = lwe * scale.fillna(1.0)
    tws_raw    = tws_scaled.resample(time='MS').mean().ffill(dim='time').bfill(dim='time')
    tws_lag1_raw  = tws_raw.shift(time=1).bfill(dim='time')
    tws_lag3_raw  = tws_raw.shift(time=3).bfill(dim='time')
    tws_lag6_raw  = tws_raw.shift(time=6).bfill(dim='time')
    tws_lag12_raw = tws_raw.shift(time=12).bfill(dim='time')
    tws_trend_6mo_raw      = tws_raw - tws_lag6_raw
    tws_trend_6mo_lag1_raw = tws_lag1_raw - tws_raw.shift(time=7).bfill(dim='time')

    n_times_g  = len(years_pm)
    time_start = pd.to_datetime('2003-01-01')
    time_end   = time_start + pd.DateOffset(months=n_times_g - 1)
    tgt_lon    = xr.DataArray(lon, dims='lon')
    tgt_lat    = xr.DataArray(lat, dims='lat')

    def _interp_grace(ts):
        return ts.sel(time=slice(time_start, time_end)).interp(
            lat=tgt_lat, lon=tgt_lon, method='nearest').values

    _PRETEST_T = min(108, n_times_g)
    def _zscore_px_pretest(flat_arr):
        arr2d = flat_arr.reshape(n_times_g, -1)
        mu = arr2d[:_PRETEST_T].mean(axis=0, keepdims=True)
        si = arr2d[:_PRETEST_T].std(axis=0, keepdims=True) + 1e-8
        return ((arr2d - mu) / si).reshape(-1).astype(np.float32)

    _g_names = [
        'grace_tws_lag0', 'grace_tws_lag1', 'grace_tws_lag3',
        'grace_tws_lag6', 'grace_tws_lag12',
        'grace_tws_trend_6mo', 'grace_tws_trend_6mo_lag1',
    ]
    _g_raws = [
        tws_raw, tws_lag1_raw, tws_lag3_raw,
        tws_lag6_raw, tws_lag12_raw,
        tws_trend_6mo_raw, tws_trend_6mo_lag1_raw,
    ]
    _g_cols, _g_added = [], []
    for _gn, _gr in zip(_g_names, _g_raws):
        if _gn not in feature_names:
            _g_cols.append(_zscore_px_pretest(np.nan_to_num(_interp_grace(_gr).reshape(-1), nan=0.0)))
            _g_added.append(_gn)
    if _g_cols:
        g_feats       = np.stack(_g_cols, axis=1)
        x_flat        = np.concatenate([x_flat, g_feats], axis=1)
        feature_names = feature_names + _g_added
        print(f'GRACE TWS merged: +{len(_g_added)} features ({_g_added}) -> {x_flat.shape[1]} total')
        del g_feats
    del ds_grace, lwe, scale, tws_scaled; gc.collect()
else:
    print('GRACE TWS NetCDF not found -- continuing without external GRACE file')

n_times  = len(years_pm)
n_pixels = len(lat) * len(lon)

# lag features
print('\nConstructing explicit deep lag features (6mo and 12mo lags)...')
lag_targets = ['era5_z_500hPa_lag0', 'oni_lag0', 'era5_tp_lag0', 'era5_vpd', 'burned_area_pred']
shifts = [6, 12]

N_MONTHS = len(years_pm)
N_PIXELS = len(lat) * len(lon)

new_feats_list = []
new_names_list = []

for base_feat in lag_targets:
    if base_feat in feature_names:
        base_idx = feature_names.index(base_feat)
        base_2d  = x_flat[:, base_idx].reshape(N_MONTHS, N_PIXELS)
        for shift in shifts:
            new_name = base_feat.replace('_lag0', '') + f'_lag{shift}'
            if new_name in feature_names or new_name in new_names_list:
                continue
            shifted_2d = np.roll(base_2d, shift=shift, axis=0)
            shifted_2d[:shift, :] = 0.0
            new_feats_list.append(shifted_2d.reshape(-1, 1).astype(np.float32))
            new_names_list.append(new_name)

if 'era5_swvl3_deficit_6mo' not in feature_names and 'era5_swvl3_deficit_6mo' not in new_names_list:
    _sw_src = 'era5_swvl3_lag0' if 'era5_swvl3_lag0' in feature_names else (
              'era5_swvl1_lag0' if 'era5_swvl1_lag0' in feature_names else None)
    if _sw_src is not None:
        _sw_2d = np.nan_to_num(x_flat[:, feature_names.index(_sw_src)].reshape(N_MONTHS, N_PIXELS), nan=0.0)
        _sw_anom = np.zeros_like(_sw_2d)
        _pre_t = min(108, N_MONTHS)
        for _m in range(12):
            _m_pre = np.arange(_m, _pre_t, 12)
            _mu = np.nanmean(_sw_2d[_m_pre], axis=0, keepdims=True)
            _sd = np.nanstd(_sw_2d[_m_pre], axis=0, keepdims=True) + 1e-8
            _m_all = np.arange(_m, N_MONTHS, 12)
            _sw_anom[_m_all] = (_sw_2d[_m_all] - _mu) / _sd
        _sw_def6 = np.zeros_like(_sw_anom)
        for _t in range(N_MONTHS):
            _s0 = max(0, _t - 5)
            _sw_def6[_t] = np.sum(np.minimum(0.0, _sw_anom[_s0:_t + 1]), axis=0)
        new_feats_list.append(_sw_def6.reshape(-1, 1).astype(np.float32))
        new_names_list.append('era5_swvl3_deficit_6mo')
        print(f"Computed fallback 'era5_swvl3_deficit_6mo' from '{_sw_src}' (pre-test 2003-2011 climatology)")

_ROLLING_NPZ_COLS = [
    'era5_swvl3_deficit_6mo',
    'drought_stress_kbdi_proxy_3mo', 'drought_stress_kbdi_proxy_6mo',
    'ndvi_cumdeficit_3mo', 'ndvi_cumdeficit_6mo', 'ndvi_cumdeficit_12mo',
    'ndvi_anom_delta', 'ndvi_drought_interact',
]
if new_feats_list:
    x_flat = np.concatenate([x_flat, np.concatenate(new_feats_list, axis=1)], axis=1)
    feature_names.extend(new_names_list)
    print(f'Added {len(new_names_list)} deep-lag/fallback features: {new_names_list}')
    new_feats_list, new_names_list = [], []

for _rc in _ROLLING_NPZ_COLS:
    _rc_l1 = f'{_rc}_lag1'
    if _rc in feature_names and _rc_l1 not in feature_names:
        _rc_2d = x_flat[:, feature_names.index(_rc)].reshape(N_MONTHS, N_PIXELS)
        _rc_l1_2d = np.zeros_like(_rc_2d)
        _rc_l1_2d[1:] = _rc_2d[:-1]
        _rc_l1_2d[0]  = _rc_2d[0]
        new_feats_list.append(_rc_l1_2d.reshape(-1, 1).astype(np.float32))
        new_names_list.append(_rc_l1)

if new_feats_list:
    x_flat = np.concatenate([x_flat, np.concatenate(new_feats_list, axis=1)], axis=1)
    feature_names.extend(new_names_list)
    print(f'Added {len(new_names_list)} lag-1 antecedent rolling features: {new_names_list}')

print(f'Total features : {len(feature_names)}  |  x_flat shape: {x_flat.shape}')


Keep only active pixels and drop near-constant features.


In [ ]:
VARIANCE_THRESHOLD  = 1e-10
BA_ACTIVE_THRESHOLD = 1e-8

n_times  = len(years_pm)
n_pixels = len(lat) * len(lon)

ba_feat_candidates = [f for f in feature_names if 'burned_area' in f.lower()]
if not ba_feat_candidates:
    raise RuntimeError('burned_area_pred column missing from feature_names')

ba_col_idx = feature_names.index(ba_feat_candidates[0])

y_ba_flat  = x_flat[:, ba_col_idx].astype(np.float32)

ba_all          = y_ba_flat.reshape(n_times, n_pixels)
active_pixel_2d = np.nanmax(ba_all, axis=0) > BA_ACTIVE_THRESHOLD
active_pixel_flat = np.tile(active_pixel_2d, n_times)

valid_mask = (~np.isnan(y_ba_flat)) & active_pixel_flat

n_raw = int(valid_mask.sum())
print(f'Step 1 — pixel-level mask : {n_raw:,} rows  ({active_pixel_2d.sum():,} active pixels)')

# drop zero-variance pixels
y_post_ba    = y_ba_flat[valid_mask]
time_post_ba = time_idx[valid_mask]
lat_post_ba  = lat_flat[valid_mask]
lon_post_ba  = lon_flat[valid_mask]

pix_idx_post_ba = (np.where(valid_mask)[0]) % n_pixels
unique_pix, inv = np.unique(pix_idx_post_ba, return_inverse=True)
pix_var         = np.array([np.nanvar(y_post_ba[inv == k]) for k in range(len(unique_pix))])
row_var         = pix_var[inv]
variance_ok     = row_var > VARIANCE_THRESHOLD

final_mask = np.zeros(len(y_ba_flat), dtype=bool)
activity_indices = np.where(valid_mask)[0]
final_mask[activity_indices[variance_ok]] = True

n_final   = int(final_mask.sum())
n_dropped = n_raw - n_final
print(f'Step 2 — variance filter  : {n_final:,} rows kept, {n_dropped:,} zero-variance rows dropped')

x_valid      = x_flat[final_mask]
time_valid   = time_idx[final_mask]
lat_valid    = lat_flat[final_mask]
lon_valid    = lon_flat[final_mask]

print(f'\nFinal dataset summary')
print(f'  Rows     : {n_final:,}')
print(f'  Features : {x_valid.shape[1]}')
y_ba_valid = x_valid[:, ba_col_idx]
print(f'  BA > 0    : {(y_ba_valid>0).mean()*100:.1f}%')


Crop to the study region and define the regional masks.


In [ ]:
ACTIVE_REGION = 'Mainland'
print(f"Modeling Active Region: {ACTIVE_REGION}")

peat_candidates = [f for f in feature_names if 'peat' in f.lower()]
if peat_candidates:
    peat_idx = feature_names.index(peat_candidates[0])
else:
    raise RuntimeError('peat_mask feature not found')

is_peat_valid = x_valid[:, peat_idx] > 0.5

idn_box = (lat_valid >= -6.5) & (lat_valid <= 7.0) & (lon_valid >= 94.5) & (lon_valid <= 119.6)

EXCLUDE_COUNTRIES = {'China', 'Taiwan', 'India', 'Bangladesh'}
try:
    import cartopy.io.shapereader as shpreader
    import shapely
except ImportError:
    import subprocess, sys as _sys
    for _pkg in ('cartopy', 'shapely'):
        subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', _pkg])
    import cartopy.io.shapereader as shpreader
    import shapely

try:
    _shp_path = shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
    _reader = shpreader.Reader(_shp_path)
    _excl_geoms = [
        rec.geometry for rec in _reader.records()
        if rec.attributes.get('NAME', rec.attributes.get('ADMIN', '')) in EXCLUDE_COUNTRIES
    ]
    if not _excl_geoms:
        raise RuntimeError(f'No geometries matched {EXCLUDE_COUNTRIES} in Natural Earth admin_0_countries')
    _excl_union = shapely.union_all(_excl_geoms).buffer(0.15)
    _is_excl_valid = shapely.contains(_excl_union, shapely.points(lon_valid, lat_valid))
    print(f'[Country exclusion] {_is_excl_valid.sum():,} / {len(lat_valid):,} rows fall inside {EXCLUDE_COUNTRIES} '
          f'and will be excluded from Mainland.')
except Exception as _e:
    print(f'[Country exclusion] WARNING: Natural Earth lookup failed ({_e}); falling back to coordinate bounding boxes.')
    _is_china_box = ((lat_valid > 21.0) & (lon_valid >= 97.0) & (lon_valid <= 110.0) & (lat_valid <= 29.5)
                     & ~((lat_valid <= 23.5) & (lon_valid <= 101.0)))
    _is_india_bgd_box = ((lat_valid > 20.0) & (lon_valid < 92.8)) | ((lat_valid > 22.0) & (lon_valid < 94.5))
    _is_excl_valid = _is_china_box | _is_india_bgd_box

mainland_box = (lat_valid > 7.0) & (lon_valid <= 110.0) & (~_is_excl_valid)

masks = {
    'Indo_Peat': is_peat_valid & idn_box,
    'Indo_NonPeat': (~is_peat_valid) & idn_box,
    'Mainland': mainland_box,
    'Rest_Of_SEAsia': ~(is_peat_valid & idn_box) & ~((~is_peat_valid) & idn_box) & ~(mainland_box)
}

idn_mask = masks[ACTIVE_REGION]

x_idn         = x_valid[idn_mask]
time_idn      = time_valid[idn_mask]
lat_idn       = lat_valid[idn_mask]
lon_idn       = lon_valid[idn_mask]

pixel_coords             = np.stack([lat_idn, lon_idn], axis=1)
unique_pixels, pixel_inv = np.unique(pixel_coords, axis=0, return_inverse=True)
n_idn_pixels             = len(unique_pixels)
pix_lats                 = unique_pixels[:, 0].astype(np.float64)
pix_lons                 = unique_pixels[:, 1].astype(np.float64)

print(f'Cropped to {ACTIVE_REGION}:')
print(f'  Rows   : {len(x_idn):,}')
print(f'  Pixels : {n_idn_pixels:,}')
print(f'  Lat    : {lat_idn.min():.2f} - {lat_idn.max():.2f}')
print(f'  Lon    : {lon_idn.min():.2f} - {lon_idn.max():.2f}')

idn_mask_in_flat = np.zeros(len(x_flat), dtype=bool)
rows_in_valid    = np.where(final_mask)[0]
idn_mask_in_flat[rows_in_valid[idn_mask]] = True
print(f'idn_mask_in_flat sum: {idn_mask_in_flat.sum():,}')


Extract the burned-area target (with unit correction).


In [ ]:
ba_col_name   = 'burned_area_pred'
ba_col_in_idn = feature_names.index(ba_col_name)

y_ba_raw = x_idn[:, ba_col_in_idn].copy().astype(np.float32)
y_ba_raw = np.nan_to_num(y_ba_raw, nan=0.0)

# convert burned area from km2 to fraction
EARTH_DEG_KM   = 111.320
pixel_area_km2 = (np.cos(np.deg2rad(lat_idn)) *
                  (0.25 * EARTH_DEG_KM) ** 2).astype(np.float32)

y_ba_idn = np.maximum(0.0, y_ba_raw / pixel_area_km2)

print('── BA target extraction + unit correction ──────────────────────────────')
print(f'  Raw BA max              : {y_ba_raw.max():.4f}  (m² per grid cell)')
print(f'  BA max (corrected)      : {y_ba_idn.max():.6f}   ← must be ≤ 1.0')
print(f'  BA p99 (fire months)    : {np.percentile(y_ba_idn[y_ba_idn>0], 99):.6f}')
print(f'  BA p95 (fire months)    : {np.percentile(y_ba_idn[y_ba_idn>0], 95):.6f}')
print(f'  BA > 0                  : {(y_ba_idn>0).mean()*100:.1f}%')
print(f'  Pixel area range (km²)  : {pixel_area_km2.min():.1f} – {pixel_area_km2.max():.1f}')

if y_ba_idn.max() > 1.0:
    raise RuntimeError(
        f'BA max={y_ba_idn.max():.4f} still > 1 after pixel-area division.\n'
        f'Check lat_idn units — expected decimal degrees, got {lat_idn[:3]}?'
    )

print()
print('y_ba_idn corrected and validated ✓')
print('Column stays in x_idn at index ba_col_in_idn for lag computation in Cell 5.')


Engineer the regional predictors.


In [ ]:
x_idn_base = x_idn.copy()

import numpy as np
import matplotlib.path as mpltPath

YEAR_START  = 2003
TEST_OFFSET = 9
YEAR_END    = 2020
TRAIN_END_T = TEST_OFFSET * 12

BASE_EXCLUDE = [
    'burned_area_pred', 'peat_mask', 'ba_nan_flag', 'lat_flat', 'lon_flat',
    'cams_tc_ch4', 'gfed_defo_lag0', 'gfed_defo_cumsum_3mo_lag3',
]
MAINLAND_FEATURES_BA = [f for f in feature_names if f not in BASE_EXCLUDE]
print(f'Base features from .npz + GRACE + deep lags: {len(MAINLAND_FEATURES_BA)}')

feature_names_idn_full = list(feature_names)

# fire history features
ba_col_in_idn = feature_names.index('burned_area_pred')
ba_3d = np.nan_to_num(x_flat[:, ba_col_in_idn].reshape(n_times, n_pixels), nan=0.0)
cum_ba = np.zeros_like(ba_3d)
for t in range(n_times):
    t0 = max(0, t - 24); t1 = max(0, t - 12)
    if t1 > t0:
        cum_ba[t] = ba_3d[t0:t1].sum(axis=0)
x_idn = np.concatenate([x_idn, cum_ba.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
feature_names_idn_full.append('cumulative_ba_13_24mo'); MAINLAND_FEATURES_BA.append('cumulative_ba_13_24mo')

cal_years = YEAR_START + (time_idn // 12)
ba_prev_year = np.zeros(len(y_ba_idn), dtype=np.float32)
for pid in range(n_idn_pixels):
    pm = pixel_inv == pid
    yp_arr = y_ba_idn[pm]; yr_arr = cal_years[pm]
    ytot = {yr: float(np.sum(yp_arr[yr_arr == yr])) for yr in np.unique(yr_arr)}
    for idx in np.where(pm)[0]:
        ba_prev_year[idx] = ytot.get(cal_years[idx] - 1, 0.0)
x_idn = np.concatenate([x_idn, ba_prev_year[:, None].astype(np.float32)], axis=1)
feature_names_idn_full.append('ba_prev_year'); MAINLAND_FEATURES_BA.append('ba_prev_year')

# drought and rainfall deficit features
if 'era5_vpd' in feature_names:
    vpd_3d2 = x_flat[:, feature_names.index('era5_vpd')].reshape(n_times, n_pixels)
    cum_vpd = np.zeros_like(vpd_3d2)
    for yr in range(18):
        for m in range(12):
            t = yr * 12 + m
            if t >= n_times: break
            cum_vpd[t] = vpd_3d2[t] if m == 10 else (cum_vpd[t - 1] + vpd_3d2[t] if t > 0 else vpd_3d2[t])
    x_idn = np.concatenate([x_idn, cum_vpd.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('vpd_cumdeficit_since_nov'); MAINLAND_FEATURES_BA.append('vpd_cumdeficit_since_nov')

    cum_vpd_lag1_3d = np.zeros_like(cum_vpd)
    for yr in range(18):
        for m in range(12):
            t = yr * 12 + m
            if t >= n_times: break
            cum_vpd_lag1_3d[t] = 0.0 if (m == 10 or t == 0) else cum_vpd[t - 1]
    x_idn = np.concatenate([x_idn, cum_vpd_lag1_3d.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('vpd_cumdeficit_thru_lag1'); MAINLAND_FEATURES_BA.append('vpd_cumdeficit_thru_lag1')

if 'era5_tp_lag0' in feature_names and 'era5_tp_lag1' in feature_names:
    tp0 = x_idn[:, feature_names.index('era5_tp_lag0')]
    tp1 = x_idn[:, feature_names.index('era5_tp_lag1')]
    rr  = tp0 + tp1
    thresh = np.array([
        np.nanpercentile(rr[(pixel_inv == p) & (time_idn < TRAIN_END_T)], 90)
        if np.any((pixel_inv == p) & (time_idn < TRAIN_END_T)) else np.nanpercentile(rr[pixel_inv == p], 90)
        for p in range(n_idn_pixels)
    ], dtype=np.float32)
    db = (rr > thresh[pixel_inv]).astype(np.float32)
    x_idn = np.concatenate([x_idn, db[:, None]], axis=1)
    feature_names_idn_full.append('drought_breaker_flag'); MAINLAND_FEATURES_BA.append('drought_breaker_flag')

if 'era5_tp_lag1' in feature_names and 'era5_tp_lag2' in feature_names:
    tp1_ = x_idn[:, feature_names.index('era5_tp_lag1')]
    tp2_ = x_idn[:, feature_names.index('era5_tp_lag2')]
    rr_lag1 = tp1_ + tp2_
    thresh_lag1 = np.array([
        np.nanpercentile(rr_lag1[(pixel_inv == p) & (time_idn < TRAIN_END_T)], 90)
        if np.any((pixel_inv == p) & (time_idn < TRAIN_END_T)) else np.nanpercentile(rr_lag1[pixel_inv == p], 90)
        for p in range(n_idn_pixels)
    ], dtype=np.float32)
    db_lag1 = (rr_lag1 > thresh_lag1[pixel_inv]).astype(np.float32)
    x_idn = np.concatenate([x_idn, db_lag1[:, None]], axis=1)
    feature_names_idn_full.append('drought_breaker_flag_lag1'); MAINLAND_FEATURES_BA.append('drought_breaker_flag_lag1')

if 'era5_tp_lag0' in feature_names:
    tp_3d  = x_flat[:, feature_names.index('era5_tp_lag0')].reshape(n_times, len(lat), len(lon))
    tp_p10 = np.nanpercentile(tp_3d[:TRAIN_END_T], 10, axis=0)
    dsrain = np.zeros_like(tp_3d)
    for t in range(1, n_times):
        dsrain[t] = np.where(tp_3d[t] > tp_p10, 0, dsrain[t - 1] + 1)
    x_idn = np.concatenate([x_idn, dsrain.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('days_since_last_rain'); MAINLAND_FEATURES_BA.append('days_since_last_rain')
    dsrain_l1 = np.zeros_like(dsrain); dsrain_l1[1:] = dsrain[:-1]
    x_idn = np.concatenate([x_idn, dsrain_l1.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('days_since_last_rain_lag1'); MAINLAND_FEATURES_BA.append('days_since_last_rain_lag1')

if 'era5_tp_lag1' in feature_names:
    tp1_idx = feature_names.index('era5_tp_lag1')
    _monsoon_mask = np.isin(time_idn % 12, [6, 7])
    monsoon_precip = x_idn[:, tp1_idx].copy()
    monsoon_precip[~_monsoon_mask] = 0.0
    x_idn = np.concatenate([x_idn, monsoon_precip[:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('monsoon_onset_tp_sepoct'); MAINLAND_FEATURES_BA.append('monsoon_onset_tp_sepoct')

if 'static_soc' in feature_names:
    sc = x_idn[:, feature_names.index('static_soc')].copy()
    _tr_rows = time_idn < TRAIN_END_T
    sc_z = ((sc - np.nanmean(sc[_tr_rows])) / (np.nanstd(sc[_tr_rows]) + 1e-8)).astype(np.float32)
    x_idn = np.concatenate([x_idn, sc_z[:, None]], axis=1)
    feature_names_idn_full.append('soc_domain_zscore'); MAINLAND_FEATURES_BA.append('soc_domain_zscore')
    if 'vpd_cumdeficit_since_nov' in feature_names_idn_full:
        fp = sc_z * np.maximum(0.0, x_idn[:, feature_names_idn_full.index('vpd_cumdeficit_since_nov')])
        x_idn = np.concatenate([x_idn, fp[:, None].astype(np.float32)], axis=1)
        feature_names_idn_full.append('soc_x_vpd_cumdeficit'); MAINLAND_FEATURES_BA.append('soc_x_vpd_cumdeficit')
    if 'vpd_cumdeficit_thru_lag1' in feature_names_idn_full:
        fp_l1 = sc_z * np.maximum(0.0, x_idn[:, feature_names_idn_full.index('vpd_cumdeficit_thru_lag1')])
        x_idn = np.concatenate([x_idn, fp_l1[:, None].astype(np.float32)], axis=1)
        feature_names_idn_full.append('soc_x_vpd_cumdeficit_lag1'); MAINLAND_FEATURES_BA.append('soc_x_vpd_cumdeficit_lag1')

if 'era5_swvl3_deficit_6mo' in feature_names_idn_full and 'oni_lag0' in feature_names_idn_full:
    _sw3_col = x_idn[:, feature_names_idn_full.index('era5_swvl3_deficit_6mo')]
    _oni0_col = x_idn[:, feature_names_idn_full.index('oni_lag0')]
    sw3_x_oni = (_sw3_col * np.maximum(0.0, _oni0_col)).astype(np.float32)
    x_idn = np.concatenate([x_idn, sw3_x_oni[:, None]], axis=1)
    feature_names_idn_full.append('swvl3_deficit_x_oni'); MAINLAND_FEATURES_BA.append('swvl3_deficit_x_oni')
if 'era5_swvl3_deficit_6mo_lag1' in feature_names_idn_full and 'oni_lag1' in feature_names_idn_full:
    _sw3_l1 = x_idn[:, feature_names_idn_full.index('era5_swvl3_deficit_6mo_lag1')]
    _oni1_col = x_idn[:, feature_names_idn_full.index('oni_lag1')]
    sw3_x_oni_l1 = (_sw3_l1 * np.maximum(0.0, _oni1_col)).astype(np.float32)
    x_idn = np.concatenate([x_idn, sw3_x_oni_l1[:, None]], axis=1)
    feature_names_idn_full.append('swvl3_deficit_x_oni_lag1'); MAINLAND_FEATURES_BA.append('swvl3_deficit_x_oni_lag1')

fire_freq = np.zeros(n_idn_pixels, dtype=np.float32)
yrs_since = np.zeros(n_idn_pixels, dtype=np.float32)
for pid in range(n_idn_pixels):
    pm = pixel_inv == pid
    tr = pm & (time_idn < TRAIN_END_T)
    yb = y_ba_idn[tr]; tb = time_idn[tr]
    fm = tb[yb > 1e-6]
    fire_freq[pid] = len(fm) / max(tr.sum(), 1)
    yrs_since[pid] = (TRAIN_END_T - fm.max()) / 12.0 if len(fm) > 0 else 10.0
x_idn = np.concatenate([x_idn, fire_freq[pixel_inv][:, None], yrs_since[pixel_inv][:, None]], axis=1)
feature_names_idn_full += ['fire_freq_train', 'years_since_last_fire']
MAINLAND_FEATURES_BA   += ['fire_freq_train', 'years_since_last_fire']

DMI_JULY_BY_YEAR = {
    2003: 0.1,  2004: 0.3,  2005: 0.1,  2006: 0.7,  2007: 0.8,
    2008: 0.0,  2009: 0.2,  2010:-0.4,  2011: 0.2,  2012: 0.2,
    2013: 0.3,  2014: 0.5,  2015: 0.3,  2016: 0.1,  2017:-0.1,
    2018: 0.1,  2019: 1.5,  2020: 0.3,
}
_cal_year_idn = YEAR_START + (time_idn // 12)
dmi_jul_col = np.array([DMI_JULY_BY_YEAR.get(int(yr), 0.0) for yr in _cal_year_idn], dtype=np.float32)
x_idn = np.concatenate([x_idn, dmi_jul_col[:, None]], axis=1)
feature_names_idn_full.append('dmi_jul'); MAINLAND_FEATURES_BA.append('dmi_jul')

oni_col_idx_ext_diag = feature_names_idn_full.index('oni_lag0') if 'oni_lag0' in feature_names_idn_full else None
if oni_col_idx_ext_diag is not None:
    oni_now_col = x_idn[:, oni_col_idx_ext_diag]
    oni_x_dmi_col = (np.maximum(0.0, oni_now_col) * np.maximum(0.0, dmi_jul_col)).astype(np.float32)
    x_idn = np.concatenate([x_idn, oni_x_dmi_col[:, None]], axis=1)
    feature_names_idn_full.append('oni_x_dmi'); MAINLAND_FEATURES_BA.append('oni_x_dmi')

if 'oni_lag1' in feature_names_idn_full:
    oni_l1_col = x_idn[:, feature_names_idn_full.index('oni_lag1')]
    oni_x_dmi_l1 = (np.maximum(0.0, oni_l1_col) * np.maximum(0.0, dmi_jul_col)).astype(np.float32)
    x_idn = np.concatenate([x_idn, oni_x_dmi_l1[:, None]], axis=1)
    feature_names_idn_full.append('oni_x_dmi_lag1'); MAINLAND_FEATURES_BA.append('oni_x_dmi_lag1')

# sub-region masks
myanmar_poly = [
    (92.0, 28.5), (92.0, 9.0), (98.5, 9.0), (98.6, 11.5), (98.5, 14.5),
    (99.0, 16.5), (97.8, 18.5), (98.5, 19.5), (101.2, 21.0), (99.0, 24.0), (98.0, 28.5)
]
_path = mpltPath.Path(myanmar_poly)
_pts  = np.vstack((lon_idn, lat_idn)).T
is_myanmar_west = _path.contains_points(_pts).astype(np.float32)
is_lower_mekong = (~_path.contains_points(_pts)).astype(np.float32)

_poly_pixel_labels = _path.contains_points(
    np.vstack((unique_pixels[:, 1], unique_pixels[:, 0])).T
).astype(np.float32)
is_myanmar_west_pixel = _poly_pixel_labels
is_lower_mekong_pixel = 1.0 - is_myanmar_west_pixel

VN_NS_SPLIT_LAT = 16.5
TH_NS_SPLIT_LAT = 15.5
_pix_lat = unique_pixels[:, 0]
_pix_lon = unique_pixels[:, 1]
_is_cambodia_pix = (_pix_lat >= 10.0) & (_pix_lat <= 14.5) & (_pix_lon >= 102.3) & (_pix_lon <= 107.6)
_is_laos_pix     = (_pix_lat > 14.5) & (_pix_lat <= 22.5) & (_pix_lon >= 100.0) & (_pix_lon <= 107.8) & ~((_pix_lat < VN_NS_SPLIT_LAT) & (_pix_lon >= 106.5))
_ns_split_pix    = np.where(_pix_lon >= 105.5, VN_NS_SPLIT_LAT, TH_NS_SPLIT_LAT)
is_northern_lower_mekong_pixel = ((is_myanmar_west_pixel < 0.5) & ((_pix_lat >= _ns_split_pix) | _is_laos_pix) & (~_is_cambodia_pix)).astype(np.float32)
is_southern_lower_mekong_pixel = ((is_myanmar_west_pixel < 0.5) & (is_northern_lower_mekong_pixel < 0.5)).astype(np.float32)

is_northern_lower_mekong = is_northern_lower_mekong_pixel[pixel_inv].astype(np.float32)
is_southern_lower_mekong = is_southern_lower_mekong_pixel[pixel_inv].astype(np.float32)

region_cluster_pixel_flags = {
    'Myanmar_West':          is_myanmar_west_pixel,
    'Northern_Lower_Mekong': is_northern_lower_mekong_pixel,
    'Southern_Lower_Mekong': is_southern_lower_mekong_pixel,
}
print(f"[Mainland 3-region split] Myanmar_West={int(is_myanmar_west_pixel.sum())}, "
      f"Northern_Lower_Mekong={int(is_northern_lower_mekong_pixel.sum())}, "
      f"Southern_Lower_Mekong={int(is_southern_lower_mekong_pixel.sum())} pixels")

feature_idx_ext = {f: i for i, f in enumerate(feature_names_idn_full)}


Build lagged deforestation and cropland fire-driver features.


In [ ]:
_defo_crop_lag_names = (
    [f'defo_ba_lag{l}' for l in [3, 6, 12, 18, 24]] +
    ['defo_ba_cumulative_13_24mo', 'years_since_first_defo'] +
    [f'crop_ba_lag{l}' for l in [1, 2, 3, 6, 12, 18, 24]]
)
_present = [f for f in _defo_crop_lag_names if f in feature_names]
print(f'Defo/Crop base lag features present in .npz: {len(_present)}/{len(_defo_crop_lag_names)}')

if 'defo_ba_lag3' in feature_names_idn_full and 'defo_ba_lag6' in feature_names_idn_full:
    defo_recent = (x_idn[:, feature_names_idn_full.index('defo_ba_lag3')] +
                   x_idn[:, feature_names_idn_full.index('defo_ba_lag6')])
    x_idn = np.concatenate([x_idn, defo_recent[:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('defo_recent_pressure_3_6mo')
    MAINLAND_FEATURES_BA.append('defo_recent_pressure_3_6mo')

if all(f'crop_ba_lag{l}' in feature_names_idn_full for l in [1, 2, 3]):
    crop_recent = sum(x_idn[:, feature_names_idn_full.index(f'crop_ba_lag{l}')] for l in [1, 2, 3])
    x_idn = np.concatenate([x_idn, crop_recent[:, None].astype(np.float32)], axis=1)
    feature_names_idn_full.append('crop_recent_pressure_1_3mo')
    MAINLAND_FEATURES_BA.append('crop_recent_pressure_1_3mo')

_need_cum = any(
    f'{p}_cumulative_{w}mo' not in feature_names_idn_full
    for p in ('defo_ba', 'crop_ba') for w in (9, 12)
)
if _need_cum and 'GFED_TARGET_PATH' in dir() and os.path.exists(GFED_TARGET_PATH):
    _gfed_reload = np.load(GFED_TARGET_PATH)
    for _skey, _pref in [('y_defo_flat', 'defo_ba'), ('y_crop_flat', 'crop_ba')]:
        if _skey in _gfed_reload:
            _s3d = np.nan_to_num(_gfed_reload[_skey].astype(np.float32).reshape(n_times, n_pixels), nan=0.0)
            for _w in (9, 12):
                _cname = f'{_pref}_cumulative_{_w}mo'
                if _cname not in feature_names_idn_full:
                    _cum = np.zeros_like(_s3d)
                    for _t in range(1, n_times):
                        _s0 = max(0, _t - _w)
                        _cum[_t] = _s3d[_s0:_t].sum(axis=0)
                    x_idn = np.concatenate([x_idn, _cum.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
                    feature_names_idn_full.append(_cname)
                    MAINLAND_FEATURES_BA.append(_cname)
    del _gfed_reload; gc.collect()

feature_idx_ext = {f: i for i, f in enumerate(feature_names_idn_full)}
print(f'Mainland feature matrix ready: x_idn={x_idn.shape}, candidate features={len(MAINLAND_FEATURES_BA)}')


Apply per-pixel cube-root normalisation to the target.


In [ ]:
assert y_ba_idn.max() <= 1.0, f'BA max={y_ba_idn.max():.4f} > 1'
SCALE_BA = 1e3
y_ba_cbrt      = np.cbrt(y_ba_idn * SCALE_BA)
pixel_stats_ba = {}
y_ba_norm      = np.zeros_like(y_ba_cbrt)

for pid in range(n_idn_pixels):
    prows = pixel_inv == pid
    tr_p  = prows & (time_idn < TRAIN_END_T)
    ref_p = tr_p if tr_p.sum() >= 12 else prows
    m = float(np.nanmean(y_ba_cbrt[ref_p]))
    s = float(np.nanstd(y_ba_cbrt[ref_p])) + 1e-8
    pixel_stats_ba[pid] = (m, s)
    y_ba_norm[prows] = (y_ba_cbrt[prows] - m) / s

print(f'BA normalization complete (pre-test 2003-2011 stats): range [{y_ba_norm.min():.2f}, {y_ba_norm.max():.2f}]')


Compute per-pixel burned-area quintile thresholds.


In [ ]:
MIN_TRAIN_MONTHS = 24
_pix_mean_ba_all = []
for _pid in range(n_idn_pixels):
    _pm = (pixel_inv == _pid) & (time_idn < TRAIN_END_T)
    _yt = y_ba_idn[_pm]
    if len(_yt) >= MIN_TRAIN_MONTHS:
        _pix_mean_ba_all.append(float(np.mean(_yt)))

_pix_mean_ba_all = np.array(_pix_mean_ba_all, dtype=np.float32)
_ba_q20 = float(np.quantile(_pix_mean_ba_all, 0.20))
_ba_q40 = float(np.quantile(_pix_mean_ba_all, 0.40))
_ba_q60 = float(np.quantile(_pix_mean_ba_all, 0.60))
_ba_q80 = float(np.quantile(_pix_mean_ba_all, 0.80))
print(f'Pre-test BA quintiles: Q20={_ba_q20:.6f} Q40={_ba_q40:.6f} Q60={_ba_q60:.6f} Q80={_ba_q80:.6f}')


Train the pooled regional models: feature pruning, Optuna tuning, hurdle gate and walk-forward validation.


In [ ]:
import gc, sys, os, subprocess, pickle, warnings, multiprocessing
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.path as mpltPath
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, f1_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

_N_CPU = max(1, multiprocessing.cpu_count())
try:
    _xgb_major = int(xgb.__version__.split('.')[0])
except Exception:
    _xgb_major = 1
if 'GPU_AVAILABLE' not in dir():
    try:
        GPU_AVAILABLE = subprocess.run(['nvidia-smi'], capture_output=True, text=True).returncode == 0
    except Exception:
        GPU_AVAILABLE = False

_CB_TASK_KW = dict(task_type='GPU', devices='0') if GPU_AVAILABLE else dict(task_type='CPU', thread_count=_N_CPU)
_XGB_DEV_KW = (dict(tree_method='hist', device='cuda') if _xgb_major >= 2
               else dict(tree_method='hist', device='cuda', predictor='gpu_predictor')) if GPU_AVAILABLE else dict(tree_method='hist', n_jobs=_N_CPU)

YEAR_START          = 2003
TEST_YEAR_START     = 2012
YEAR_END            = 2020
test_years          = list(range(2012, 2021))
EVAL_YEAR_START     = test_years[0]
EVAL_YEAR_END       = test_years[-1]
BA_ACTIVE_THRESHOLD = 1e-8
MIN_CALIB_SAMPLES   = 200

# isotonic calibration
def _fit_isotonic_tail(x_fit, y_fit, tail_pct=95.0):
    x_fit = np.asarray(x_fit, dtype=np.float64)
    y_fit = np.asarray(y_fit, dtype=np.float64)
    iso_  = IsotonicRegression(out_of_bounds='clip', y_min=0.0)
    iso_.fit(x_fit, y_fit)
    ceiling   = float(np.max(x_fit))
    ceiling_y = float(iso_.predict([ceiling])[0])
    avg_slope = ceiling_y / max(ceiling, 1e-9)
    cut       = np.percentile(x_fit, tail_pct)
    tail_mask = x_fit >= cut
    if tail_mask.sum() >= 5 and len(np.unique(x_fit[tail_mask])) > 1:
        slope = float(np.polyfit(x_fit[tail_mask], y_fit[tail_mask], 1)[0])
        slope = float(np.clip(slope, 0.0, 3.0 * max(avg_slope, 1e-6)))
    else:
        slope = 0.0

    def _predict(x_arr):
        x_arr = np.asarray(x_arr, dtype=np.float64)
        out   = iso_.predict(np.minimum(x_arr, ceiling)).astype(np.float64)
        over  = x_arr > ceiling
        if np.any(over):
            out[over] = ceiling_y + slope * (x_arr[over] - ceiling)
        return np.maximum(0.0, out).astype(np.float32)
    return _predict

def _sqrt_ba_weights(y_arr):
    y_arr = np.asarray(y_arr, dtype=np.float64)
    w = np.sqrt(np.maximum(y_arr, 0.0))
    m = float(w.mean())
    if m > 1e-12:
        w = w / m
    else:
        w = np.ones_like(y_arr)
    return np.maximum(w, 1e-3).astype(np.float32)

def _inv_rmse_weights(y_true_val, p_cb_val, p_xgb_val, p_et_val):
    rmses = []
    for p in (p_cb_val, p_xgb_val, p_et_val):
        r = float(np.sqrt(mean_squared_error(y_true_val, p)))
        rmses.append(max(r, 1e-8))
    inv = np.array([1.0 / r for r in rmses], dtype=np.float64)
    return (inv / inv.sum()).astype(np.float32)

from re import compile as _re_compile
_LAG_TOKEN_RE = _re_compile(r'lag(\d+)')
_TIME_VARYING_PREFIXES = (
    'era5_', 'ndvi_', 'evi_', 'oni', 'grace_', 'cams_', 'vpd', 'tp_', 'swvl',
    'drought_stress',
)
PAST_ONLY_WHITELIST = {'monsoon_onset_tp_sepoct'}
CONCURRENT_DERIVED_NAMES = {
    'vpd_spatial_lag', 'vpd_cumdeficit_since_jan', 'vpd_cumdeficit_since_nov',
    'soc_x_vpd_cumdeficit', 'days_since_last_rain', 'drought_breaker_flag',
    'swvl3_deficit_x_oni', 'iod_x_oni', 'oni_x_dmi', 'grace_tws_trend_6mo',
}
def is_concurrent_feature(feat):
    f = feat.lower()
    if f in PAST_ONLY_WHITELIST: return False
    if f in CONCURRENT_DERIVED_NAMES: return True
    lags = [int(k) for k in _LAG_TOKEN_RE.findall(f)]
    if any(k == 0 for k in lags): return True
    if f.startswith(_TIME_VARYING_PREFIXES) and not any(k >= 1 for k in lags): return True
    return False
is_forbidden = is_concurrent_feature
def drop_concurrent(feat_list, label='', where='', verbose=True):
    kept = [f for f in feat_list if not is_concurrent_feature(f)]
    dropped = [f for f in feat_list if is_concurrent_feature(f)]
    if verbose and dropped:
        print(f"[lead1] {label or where}: removed {len(dropped)} concurrent feature(s)")
    return kept
drop_concurrent_features = drop_concurrent
def assert_no_concurrent(feat_list, label='', where=''):
    bad = [f for f in feat_list if is_concurrent_feature(f)]
    assert not bad, f"[lead1] concurrent feature(s) in {label or where}: {bad}"

MAINLAND_FEATURES_BA = drop_concurrent(MAINLAND_FEATURES_BA, label='MAINLAND_FEATURES_BA')

myanmar_poly = [
    (92.0, 28.5), (92.0, 9.0), (98.5, 9.0), (98.6, 11.5), (98.5, 14.5),
    (99.0, 16.5), (97.8, 18.5), (98.5, 19.5), (101.2, 21.0), (99.0, 24.0), (98.0, 28.5)
]
_myan_path = mpltPath.Path(myanmar_poly)
_pts_row   = np.vstack((lon_idn, lat_idn)).T
is_myan_row = _myan_path.contains_points(_pts_row)
is_indo_row = ~is_myan_row
VN_NS_SPLIT_LAT = 16.5
TH_NS_SPLIT_LAT = 15.5
_is_cambodia_row = (lat_idn >= 10.0) & (lat_idn <= 14.5) & (lon_idn >= 102.3) & (lon_idn <= 107.6)
_is_laos_row     = (lat_idn > 14.5) & (lat_idn <= 22.5) & (lon_idn >= 100.0) & (lon_idn <= 107.8) & ~((lat_idn < VN_NS_SPLIT_LAT) & (lon_idn >= 106.5))
_ns_split_row    = np.where(lon_idn >= 105.5, VN_NS_SPLIT_LAT, TH_NS_SPLIT_LAT)
is_indn_row = is_indo_row & ((lat_idn >= _ns_split_row) | _is_laos_row) & (~_is_cambodia_row)
is_inds_row = is_indo_row & (~is_indn_row)

_pts_pix    = np.vstack((unique_pixels[:, 1], unique_pixels[:, 0])).T
is_myan_pixel = _myan_path.contains_points(_pts_pix)
_pix_lat    = unique_pixels[:, 0]
_pix_lon    = unique_pixels[:, 1]
_is_cambodia_pix = (_pix_lat >= 10.0) & (_pix_lat <= 14.5) & (_pix_lon >= 102.3) & (_pix_lon <= 107.6)
_is_laos_pix     = (_pix_lat > 14.5) & (_pix_lat <= 22.5) & (_pix_lon >= 100.0) & (_pix_lon <= 107.8) & ~((_pix_lat < VN_NS_SPLIT_LAT) & (_pix_lon >= 106.5))
_ns_split_pix    = np.where(_pix_lon >= 105.5, VN_NS_SPLIT_LAT, TH_NS_SPLIT_LAT)
is_indn_pixel = (~is_myan_pixel) & ((_pix_lat >= _ns_split_pix) | _is_laos_pix) & (~_is_cambodia_pix)
is_inds_pixel = (~is_myan_pixel) & (~is_indn_pixel)

region_defs = {
    'Myanmar_West':          (is_myan_row, is_myan_pixel),
    'Northern_Lower_Mekong': (is_indn_row, is_indn_pixel),
    'Southern_Lower_Mekong': (is_inds_row, is_inds_pixel),
}
SUBREGION_FIRE_SEASON_MONTHS = {
    'Myanmar_West':          [1, 2, 3],
    'Northern_Lower_Mekong': [1, 2, 3],
    'Southern_Lower_Mekong': [1, 2, 3],
}
FIRE_SEASON_MONTHS = [1, 2, 3]

for _nm, _arr in [
    ('lat_sin', np.sin(np.deg2rad(lat_idn)).astype(np.float32)),
    ('lat_cos', np.cos(np.deg2rad(lat_idn)).astype(np.float32)),
    ('lon_sin', np.sin(np.deg2rad(lon_idn)).astype(np.float32)),
    ('lon_cos', np.cos(np.deg2rad(lon_idn)).astype(np.float32)),
]:
    if _nm not in feature_names_idn_full:
        x_idn = np.concatenate([x_idn, _arr[:, None]], axis=1)
        feature_names_idn_full.append(_nm)
    if _nm not in MAINLAND_FEATURES_BA:
        MAINLAND_FEATURES_BA.append(_nm)

_CAND_ML = [f for f in MAINLAND_FEATURES_BA if f in feature_names_idn_full and f not in ('y_lag1', 'y_lag3')]
REGION_FEAT_MAP = {
    'Myanmar_West':          list(_CAND_ML),
    'Northern_Lower_Mekong': list(_CAND_ML),
    'Southern_Lower_Mekong': list(_CAND_ML),
}
_ALL_FEAT_NAMES = feature_names_idn_full

if 'y_lag1_global' not in dir():
    y_lag1_global = np.zeros(len(y_ba_idn), dtype=np.float32)
    y_lag3_global = np.zeros(len(y_ba_idn), dtype=np.float32)
    for _pid in range(n_idn_pixels):
        _pm = pixel_inv == _pid
        _yp = y_ba_idn[_pm]; _tp = time_idn[_pm]
        _so = np.argsort(_tp); _ys = _yp[_so]
        _l1 = np.zeros_like(_ys); _l1[1:] = _ys[:-1]
        _l3 = np.zeros_like(_ys); _l3[3:] = _ys[:-3]
        _oi = np.where(_pm)[0][_so]
        y_lag1_global[_oi] = _l1; y_lag3_global[_oi] = _l3

y_lag12_global = np.zeros(len(y_ba_idn), dtype=np.float32)
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid
    _yp = y_ba_idn[_pm]; _tp = time_idn[_pm]
    _so = np.argsort(_tp); _ys = _yp[_so]
    _l12 = np.zeros_like(_ys); _l12[12:] = _ys[:-12]
    _oi = np.where(_pm)[0][_so]
    y_lag12_global[_oi] = _l12

n_rows = len(y_ba_idn)
y_pred_final_ba     = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_prehurdle = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_raw_ens      = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_ridge     = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_cb        = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_xgb       = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_et        = np.full(n_rows, np.nan, dtype=np.float32)
fire_prob_all       = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_hurdle_prob  = np.full(n_rows, np.nan, dtype=np.float32)
y_fire_prob         = np.full(n_rows, np.nan, dtype=np.float32)
y_bin_all           = (y_ba_idn > BA_ACTIVE_THRESHOLD).astype(np.float32)
y_true_fire_bin     = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_fire_bin     = np.full(n_rows, np.nan, dtype=np.float32)
y_used_hurdle       = np.zeros(n_rows, dtype=bool)
y_hurdle_zeroed     = np.zeros(n_rows, dtype=bool)
y_train_max_ba      = np.full(n_rows, np.nan, dtype=np.float32)
y_raw_ens_max_ba    = np.full(n_rows, np.nan, dtype=np.float32)

region_hurdle_gate              = {}
region_hurdle_gates             = {}
region_hyperparams              = {}
TUNED_HYPERPARAMS               = {}
region_feature_importance       = {}
REGION_FEATURE_IMPORTANCE       = {}
all_region_feat_importance      = {}
conformal_margins               = {}
conformal_margin_by_fold_domain = {}
non_fire_season_fpr             = {}

def _json_safe(v):
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    return str(v)

print(f"\n{'='*80}")
print(" POOLED REGIONAL WALK-FORWARD TRAINING — MAINLAND SOUTHEAST ASIA (LEAD-1 FORECASTER)")
print(f"{'='*80}\n")

for reg_name, _reg_def in region_defs.items():
    row_mask = _reg_def[0] if isinstance(_reg_def, tuple) else _reg_def
    print(f"\n{'='*60}")
    print(f" REGION: {reg_name}")
    print(f"{'='*60}")

    n_reg_rows = int(row_mask.sum())
    if n_reg_rows < 50:
        print(f"[{reg_name}] Insufficient rows ({n_reg_rows}) — skipping.")
        continue

    reg_fire_months = SUBREGION_FIRE_SEASON_MONTHS[reg_name]
    cand_feats = [f for f in REGION_FEAT_MAP[reg_name] if f in _ALL_FEAT_NAMES]
    cand_feats = drop_concurrent_features(cand_feats, where=f'{reg_name} candidate pool', verbose=False)
    assert_no_concurrent(cand_feats, where=f'{reg_name} candidate pool')
    cand_idx = [_ALL_FEAT_NAMES.index(f) for f in cand_feats]
    global_rows = np.where(row_mask)[0]
    X_cand      = x_idn[row_mask][:, cand_idx].astype(np.float32)
    y_reg       = y_ba_idn[row_mask].astype(np.float32)
    t_reg       = time_idn[row_mask]
    pidinv_reg  = pixel_inv[row_mask]
    cal_yr_reg  = YEAR_START + (t_reg // 12)
    y_bin_reg   = (y_reg > BA_ACTIVE_THRESHOLD).astype(np.float32)
    _pretest_m  = cal_yr_reg < TEST_YEAR_START

    # feature selection
    _col_std    = np.nanstd(X_cand[_pretest_m], axis=0)
    _active_pos = np.where(_col_std >= 1e-10)[0]
    cand_feats  = [cand_feats[i] for i in _active_pos]
    cand_idx    = [cand_idx[i] for i in _active_pos]
    X_cand      = X_cand[:, _active_pos]

    _val_yrs_pre = [2009, 2010, 2011]
    imp_accum_pre = np.zeros(len(cand_feats), dtype=np.float64)
    imp_folds_pre = 0
    for _vyr in _val_yrs_pre:
        _tr_p = cal_yr_reg < _vyr
        _va_p = (cal_yr_reg == _vyr) & np.isin(t_reg % 12, reg_fire_months)
        if _va_p.sum() < 15:
            _va_p = (cal_yr_reg == _vyr)
        if _tr_p.sum() < 100 or _va_p.sum() < 10:
            continue
        _proxy = xgb.XGBRegressor(
            n_estimators=100, learning_rate=0.08, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0, **_XGB_DEV_KW
        )
        _proxy.fit(X_cand[_tr_p], y_reg[_tr_p])
        _perm_r = permutation_importance(
            _proxy, X_cand[_va_p], y_reg[_va_p], n_repeats=3, random_state=42, n_jobs=_N_CPU
        )
        imp_accum_pre += _perm_r.importances_mean
        imp_folds_pre += 1
    imp_mean_pre = (imp_accum_pre / imp_folds_pre) if imp_folds_pre > 0 else np.ones(len(cand_feats))

    _X_pre = X_cand[_pretest_m]
    _rng   = np.random.default_rng(42)
    _samp  = _rng.choice(len(_X_pre), size=min(8000, len(_X_pre)), replace=False)
    if len(cand_feats) > 2:
        _rho, _ = spearmanr(_X_pre[_samp], axis=0)
        _rho = np.atleast_2d(np.nan_to_num(_rho, nan=0.0, posinf=1.0, neginf=-1.0))
        np.fill_diagonal(_rho, 1.0)
        _dist = np.clip(1.0 - np.abs(np.clip(_rho, -1.0, 1.0)), 0.0, 2.0)
        _dist = (_dist + _dist.T) / 2.0
        np.fill_diagonal(_dist, 0.0)
        _link = linkage(squareform(_dist, checks=False), method='complete')
        _clusters = fcluster(_link, 0.30, criterion='distance')
    else:
        _clusters = np.arange(len(cand_feats))

    _stage1_pos = []
    for _cid in np.unique(_clusters):
        _c_members = np.where(_clusters == _cid)[0]
        _best_m = _c_members[int(np.argmax(imp_mean_pre[_c_members]))]
        _stage1_pos.append(int(_best_m))
    _stage1_pos = sorted(_stage1_pos)

    _stage2_pos = [i for i in _stage1_pos if imp_mean_pre[i] > 0.0]
    if len(_stage2_pos) < 5:
        _stage2_pos = sorted(_stage1_pos, key=lambda i: imp_mean_pre[i], reverse=True)[:max(5, len(_stage1_pos))]
        _stage2_pos = sorted(_stage2_pos)

    reg_feats    = [cand_feats[i] for i in _stage2_pos]
    reg_feat_idx = [cand_idx[i] for i in _stage2_pos]
    merged_feats = list(reg_feats)
    reg_feat_names = list(reg_feats)
    X_reg        = X_cand[:, _stage2_pos].astype(np.float32)
    print(f"[{reg_name}] Two-stage feature selection (pre-test 2003-2011): "
          f"{len(cand_feats)} candidates -> {len(_stage1_pos)} after Spearman |r|>0.70 -> "
          f"{len(reg_feats)} after >0 permutation importance.")

    _active_frac_reg      = float((y_reg[_pretest_m] > BA_ACTIVE_THRESHOLD).mean())
    REGION_QUANTILE_ALPHA = float(np.clip(0.65 - 0.5 * _active_frac_reg, 0.55, 0.65))
    print(f"[{reg_name}] Pre-test active-fire fraction={_active_frac_reg:.3f} -> Quantile alpha={REGION_QUANTILE_ALPHA:.2f}")

    # optuna tuning
    _unique_p = np.unique(pidinv_reg[_pretest_m])
    _p_means  = np.array([
        float(np.mean(y_reg[(pidinv_reg == p) & _pretest_m])) for p in _unique_p
    ], dtype=np.float32)
    _q_edges  = np.quantile(_p_means, [0.2, 0.4, 0.6, 0.8]) if len(_unique_p) >= 5 else np.array([0.0])
    _p_bins   = np.digitize(_p_means, _q_edges)
    n_samp    = max(15, int(np.ceil(len(_unique_p) * 0.15)))
    if len(_unique_p) > n_samp:
        _sampled_p = []
        for _b in range(5):
            _bp = _unique_p[_p_bins == _b]
            if len(_bp) == 0:
                continue
            _nb = max(1, int(np.round(n_samp * len(_bp) / len(_unique_p))))
            _sampled_p.append(_rng.choice(_bp, size=min(len(_bp), _nb), replace=False))
        _s_p = np.concatenate(_sampled_p) if _sampled_p else _unique_p
    else:
        _s_p = _unique_p

    _opt_rows = _pretest_m & np.isin(pidinv_reg, _s_p)
    X_opt     = X_reg[_opt_rows]
    y_opt     = y_reg[_opt_rows]
    yr_opt    = cal_yr_reg[_opt_rows]
    mo_opt    = (t_reg[_opt_rows] % 12)
    opt_w     = _sqrt_ba_weights(y_opt)

    _opt_splits = []
    for _vyr in [2009, 2010, 2011]:
        _tr_i = np.where(yr_opt < _vyr)[0]
        _va_i = np.where((yr_opt == _vyr) & np.isin(mo_opt, reg_fire_months))[0]
        if len(_va_i) < 10:
            _va_i = np.where(yr_opt == _vyr)[0]
        if len(_tr_i) >= 50 and len(_va_i) >= 5 and np.var(y_opt[_va_i]) > 1e-16:
            _opt_splits.append((_tr_i, _va_i))

    def cb_obj(trial):
        p = dict(
            iterations=200,
            depth=trial.suggest_int('depth', 4, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1.0, 30.0),
            loss_function=f'Quantile:alpha={REGION_QUANTILE_ALPHA}',
            random_seed=42, verbose=0, **_CB_TASK_KW,
        )
        if not _opt_splits:
            return 0.0
        scores = []
        for _tr_i, _va_i in _opt_splits:
            m = CatBoostRegressor(**p).fit(X_opt[_tr_i], y_opt[_tr_i], sample_weight=opt_w[_tr_i])
            pred = np.maximum(0.0, m.predict(X_opt[_va_i]).astype(np.float32))
            scores.append(float(r2_score(y_opt[_va_i], pred)))
        return float(np.mean(scores))

    def xgb_obj(trial):
        p = dict(
            n_estimators=150,
            max_depth=trial.suggest_int('max_depth', 4, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            objective='reg:quantileerror', quantile_alpha=REGION_QUANTILE_ALPHA,
            random_state=42, verbosity=0, **_XGB_DEV_KW,
        )
        if not _opt_splits:
            return 0.0
        scores = []
        for _tr_i, _va_i in _opt_splits:
            m = xgb.XGBRegressor(**p).fit(X_opt[_tr_i], y_opt[_tr_i], sample_weight=opt_w[_tr_i])
            pred = np.maximum(0.0, m.predict(X_opt[_va_i]).astype(np.float32))
            scores.append(float(r2_score(y_opt[_va_i], pred)))
        return float(np.mean(scores))

    nt = max(10, min(25, int(len(_unique_p) / 40) + 10))
    cb_study = optuna.create_study(direction='maximize')
    cb_study.optimize(cb_obj, n_trials=nt)
    CB_PARAMS = dict(
        cb_study.best_params, iterations=1000,
        loss_function=f'Quantile:alpha={REGION_QUANTILE_ALPHA}',
        random_seed=42, verbose=0, **_CB_TASK_KW,
    )

    xgb_study = optuna.create_study(direction='maximize')
    xgb_study.optimize(xgb_obj, n_trials=nt)
    XGB_PARAMS = dict(
        xgb_study.best_params, n_estimators=600,
        objective='reg:quantileerror', quantile_alpha=REGION_QUANTILE_ALPHA,
        random_state=42, verbosity=0, **_XGB_DEV_KW,
    )
    ET_PARAMS = dict(n_estimators=400, max_depth=None, min_samples_leaf=20, n_jobs=_N_CPU, random_state=42)

    def _select_hurdle_threshold(probs_val, ens_val, y_val, yb_val):
        cand_grid = np.round(np.arange(0.05, 0.65, 0.05), 2)
        valid_cands = []
        best_f2_fallback, best_thr_fallback = -1.0, 0.30
        for thr in cand_grid:
            yh   = (probs_val >= thr).astype(int)
            tp   = int(np.sum((yh == 1) & (yb_val == 1)))
            fp   = int(np.sum((yh == 1) & (yb_val == 0)))
            fn   = int(np.sum((yh == 0) & (yb_val == 1)))
            prec = tp / max(tp + fp, 1)
            rec  = tp / max(tp + fn, 1)
            f2   = (5.0 * prec * rec) / max(4.0 * prec + rec, 1e-9)
            if f2 > best_f2_fallback:
                best_f2_fallback, best_thr_fallback = f2, float(thr)
            if prec >= 0.30:
                gated = np.where(probs_val >= thr, ens_val, 0.0)
                val_r2 = float(r2_score(y_val, gated)) if len(y_val) > 1 and np.var(y_val) > 1e-16 else -1e9
                valid_cands.append((val_r2, f2, float(thr)))
        if valid_cands:
            valid_cands.sort(key=lambda x: (x[0], x[1]), reverse=True)
            return valid_cands[0][2], valid_cands[0][1]
        return best_thr_fallback, max(0.0, best_f2_fallback)

    # walk-forward training loop
    y_pred_reg           = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_pred_prehurdle_reg = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_pred_raw_reg       = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_cb_reg             = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_xgb_reg            = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_et_reg             = np.full(n_reg_rows, np.nan, dtype=np.float32)
    fire_prob_reg        = np.full(n_reg_rows, np.nan, dtype=np.float32)
    pred_fire_bin_reg    = np.full(n_reg_rows, np.nan, dtype=np.float32)
    used_hurdle_reg      = np.zeros(n_reg_rows, dtype=bool)
    hurdle_zeroed_reg    = np.zeros(n_reg_rows, dtype=bool)
    train_max_reg        = np.full(n_reg_rows, np.nan, dtype=np.float32)
    raw_ens_max_reg      = np.full(n_reg_rows, np.nan, dtype=np.float32)

    conf_margins     = {}
    imp_accum_f      = np.zeros(len(reg_feats), dtype=np.float64)
    imp_weight       = 0
    _calib_pred_hist = []
    _calib_true_hist = []
    _oof_resid_hist  = []
    _fold_gates      = []
    _fold_f2s        = []
    _nonfire_fp_cnt  = 0
    _nonfire_tot_cnt = 0

    _seed_tr = cal_yr_reg < 2010
    _seed_va = (cal_yr_reg == 2010) & np.isin(t_reg % 12, reg_fire_months)
    if _seed_va.sum() < 15:
        _seed_va = (cal_yr_reg == 2010)
    if _seed_tr.sum() >= 100 and _seed_va.sum() >= 10:
        _sw_s  = _sqrt_ba_weights(y_reg[_seed_tr])
        _cb_s  = CatBoostRegressor(**CB_PARAMS).fit(X_reg[_seed_tr], y_reg[_seed_tr], sample_weight=_sw_s)
        _xgb_s = xgb.XGBRegressor(**XGB_PARAMS).fit(X_reg[_seed_tr], y_reg[_seed_tr], sample_weight=_sw_s)
        _et_s  = ExtraTreesRegressor(**ET_PARAMS).fit(X_reg[_seed_tr], y_reg[_seed_tr], sample_weight=_sw_s)
        _p_cb_s  = np.maximum(0.0, _cb_s.predict(X_reg[_seed_va]).astype(np.float32))
        _p_xgb_s = np.maximum(0.0, _xgb_s.predict(X_reg[_seed_va]).astype(np.float32))
        _p_et_s  = np.maximum(0.0, _et_s.predict(X_reg[_seed_va]).astype(np.float32))
        _w_s     = _inv_rmse_weights(y_reg[_seed_va], _p_cb_s, _p_xgb_s, _p_et_s)
        _ens_s   = _w_s[0] * _p_cb_s + _w_s[1] * _p_xgb_s + _w_s[2] * _p_et_s
        _calib_pred_hist.append(_ens_s.copy())
        _calib_true_hist.append(y_reg[_seed_va].copy())
        _oof_resid_hist.append(np.abs(y_reg[_seed_va] - _ens_s).astype(np.float32))

    for t_yr in test_years:
        tr_mask         = cal_yr_reg < t_yr
        te_mask         = (cal_yr_reg == t_yr) & np.isin(t_reg % 12, reg_fire_months)
        nonfire_te_mask = (cal_yr_reg == t_yr) & (~np.isin(t_reg % 12, reg_fire_months))
        if tr_mask.sum() < 100 or te_mask.sum() == 0:
            continue

        if 'fire_freq_train' in reg_feats or 'years_since_last_fire' in reg_feats:
            _t_cut_f = (t_yr - YEAR_START) * 12
            _ff_col = reg_feats.index('fire_freq_train') if 'fire_freq_train' in reg_feats else None
            _ys_col = reg_feats.index('years_since_last_fire') if 'years_since_last_fire' in reg_feats else None
            for _pid_f in np.unique(pidinv_reg):
                _pm_f = (pidinv_reg == _pid_f)
                _tr_f = _pm_f & tr_mask
                _fm_f = t_reg[_tr_f][y_reg[_tr_f] > BA_ACTIVE_THRESHOLD]
                if _ff_col is not None:
                    X_reg[_pm_f, _ff_col] = float(len(_fm_f)) / max(int(_tr_f.sum()), 1)
                if _ys_col is not None:
                    X_reg[_pm_f, _ys_col] = float(_t_cut_f - _fm_f.max()) / 12.0 if len(_fm_f) > 0 else 10.0

        _val_yr  = t_yr - 1
        _in_tr   = cal_yr_reg < _val_yr
        _in_va   = (cal_yr_reg == _val_yr) & np.isin(t_reg % 12, reg_fire_months)
        if _in_va.sum() < 15:
            _in_va = (cal_yr_reg == _val_yr)
        if _in_tr.sum() >= 100 and _in_va.sum() >= 10:
            _sw_in  = _sqrt_ba_weights(y_reg[_in_tr])
            _cb_in  = CatBoostRegressor(**CB_PARAMS).fit(X_reg[_in_tr], y_reg[_in_tr], sample_weight=_sw_in)
            _xgb_in = xgb.XGBRegressor(**XGB_PARAMS).fit(X_reg[_in_tr], y_reg[_in_tr], sample_weight=_sw_in)
            _et_in  = ExtraTreesRegressor(**ET_PARAMS).fit(X_reg[_in_tr], y_reg[_in_tr], sample_weight=_sw_in)
            _p_cb_v  = np.maximum(0.0, _cb_in.predict(X_reg[_in_va]).astype(np.float32))
            _p_xgb_v = np.maximum(0.0, _xgb_in.predict(X_reg[_in_va]).astype(np.float32))
            _p_et_v  = np.maximum(0.0, _et_in.predict(X_reg[_in_va]).astype(np.float32))
            w_ens    = _inv_rmse_weights(y_reg[_in_va], _p_cb_v, _p_xgb_v, _p_et_v)
            _ens_v   = w_ens[0] * _p_cb_v + w_ens[1] * _p_xgb_v + w_ens[2] * _p_et_v

            _clf_in = CatBoostClassifier(
                iterations=400, depth=5, learning_rate=0.08,
                random_seed=42, verbose=0, class_weights=[1.0, 2.5], **_CB_TASK_KW
            )
            if y_bin_reg[_in_tr].sum() > 0 and y_bin_reg[_in_tr].sum() < _in_tr.sum():
                _clf_in.fit(X_reg[_in_tr], y_bin_reg[_in_tr])
                _prob_v = _clf_in.predict_proba(X_reg[_in_va])[:, 1].astype(np.float32)
            else:
                _prob_v = np.zeros(_in_va.sum(), dtype=np.float32)

            HURDLE_GATE_FOLD, fold_f2 = _select_hurdle_threshold(_prob_v, _ens_v, y_reg[_in_va], y_bin_reg[_in_va])
            _calib_pred_hist.append(_ens_v.copy())
            _calib_true_hist.append(y_reg[_in_va].copy())
            _gated_v = np.where(_prob_v >= HURDLE_GATE_FOLD, _ens_v, 0.0)
            _oof_resid_hist.append(np.abs(y_reg[_in_va] - _gated_v).astype(np.float32))
        else:
            w_ens = np.array([1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0], dtype=np.float32)
            HURDLE_GATE_FOLD, fold_f2 = 0.30, 0.0

        _fold_gates.append(HURDLE_GATE_FOLD)
        _fold_f2s.append(fold_f2)

        X_tr  = X_reg[tr_mask];  y_tr  = y_reg[tr_mask]
        X_te  = X_reg[te_mask];  y_te  = y_reg[te_mask]
        yb_tr = y_bin_reg[tr_mask]
        sample_w = _sqrt_ba_weights(y_tr)

        cb   = CatBoostRegressor(**CB_PARAMS).fit(X_tr, y_tr, sample_weight=sample_w)
        xgbm = xgb.XGBRegressor(**XGB_PARAMS).fit(X_tr, y_tr, sample_weight=sample_w)
        if len(np.unique(y_tr)) > 2:
            et = ExtraTreesRegressor(**ET_PARAMS).fit(X_tr, y_tr, sample_weight=sample_w)
            p_et_te = np.maximum(0.0, et.predict(X_te).astype(np.float32))
        else:
            et = None
            p_et_te = np.zeros(len(X_te), dtype=np.float32)

        p_cb_te  = np.maximum(0.0, cb.predict(X_te).astype(np.float32))
        p_xgb_te = np.maximum(0.0, xgbm.predict(X_te).astype(np.float32))
        ens_te   = w_ens[0] * p_cb_te + w_ens[1] * p_xgb_te + w_ens[2] * p_et_te

        _calib_n = sum(len(a) for a in _calib_pred_hist)
        if _calib_n >= MIN_CALIB_SAMPLES:
            _cal_x = np.concatenate(_calib_pred_hist)
            _cal_y = np.concatenate(_calib_true_hist)
            iso_predict    = _fit_isotonic_tail(_cal_x, _cal_y)
            ens_calibrated = iso_predict(ens_te)
            calib_status   = f"on(n={_calib_n})"
        else:
            iso_predict    = None
            ens_calibrated = ens_te.copy()
            calib_status   = f"off(n={_calib_n}<{MIN_CALIB_SAMPLES})"

        clf_fold = CatBoostClassifier(
            iterations=400, depth=5, learning_rate=0.08,
            random_seed=42, verbose=0, class_weights=[1.0, 2.5], **_CB_TASK_KW
        )
        if yb_tr.sum() > 0 and yb_tr.sum() < len(yb_tr):
            clf_fold.fit(X_tr, yb_tr)
            fire_prob = clf_fold.predict_proba(X_te)[:, 1].astype(np.float32)
        else:
            fire_prob = np.zeros(len(X_te), dtype=np.float32)

        ens_gated = ens_calibrated.copy()
        ens_gated[fire_prob < HURDLE_GATE_FOLD] = 0.0

        if nonfire_te_mask.sum() > 0:
            X_nf = X_reg[nonfire_te_mask]
            y_nf = y_reg[nonfire_te_mask]
            if yb_tr.sum() > 0 and yb_tr.sum() < len(yb_tr):
                prob_nf = clf_fold.predict_proba(X_nf)[:, 1].astype(np.float32)
            else:
                prob_nf = np.zeros(len(X_nf), dtype=np.float32)
            p_cb_nf  = np.maximum(0.0, cb.predict(X_nf).astype(np.float32))
            p_xgb_nf = np.maximum(0.0, xgbm.predict(X_nf).astype(np.float32))
            p_et_nf  = np.maximum(0.0, et.predict(X_nf).astype(np.float32)) if et is not None else np.zeros(len(X_nf), dtype=np.float32)
            ens_nf   = w_ens[0] * p_cb_nf + w_ens[1] * p_xgb_nf + w_ens[2] * p_et_nf
            if iso_predict is not None:
                ens_nf = iso_predict(ens_nf)
            ens_nf[prob_nf < HURDLE_GATE_FOLD] = 0.0
            _nonfire_fp_cnt  += int(np.sum((ens_nf > BA_ACTIVE_THRESHOLD) & (y_nf <= BA_ACTIVE_THRESHOLD)))
            _nonfire_tot_cnt += int(np.sum(y_nf <= BA_ACTIVE_THRESHOLD))

        y_pred_reg[te_mask]           = ens_gated
        y_pred_prehurdle_reg[te_mask] = ens_calibrated
        y_pred_raw_reg[te_mask]       = ens_te
        y_cb_reg[te_mask]             = p_cb_te
        y_xgb_reg[te_mask]            = p_xgb_te
        y_et_reg[te_mask]             = p_et_te
        fire_prob_reg[te_mask]        = fire_prob
        pred_fire_bin_reg[te_mask]    = (fire_prob >= HURDLE_GATE_FOLD).astype(np.float32)
        used_hurdle_reg[te_mask]      = True
        hurdle_zeroed_reg[te_mask]    = (fire_prob < HURDLE_GATE_FOLD)
        train_max_reg[te_mask]        = float(np.nanmax(y_tr)) if len(y_tr) > 0 else 0.0
        raw_ens_max_reg[te_mask]      = float(np.nanmax(ens_te)) if len(ens_te) > 0 else 0.0

        imp_accum_f += np.array(cb.get_feature_importance()) * te_mask.sum()
        imp_weight  += te_mask.sum()

        if _oof_resid_hist:
            abs_resid_tr = np.concatenate(_oof_resid_hist)
        else:
            p_tr_ens = w_ens[0] * np.maximum(0.0, cb.predict(X_tr).astype(np.float32)) + \
                       w_ens[1] * np.maximum(0.0, xgbm.predict(X_tr).astype(np.float32))
            abs_resid_tr = np.abs(y_tr - p_tr_ens)
        k_idx = min(int(np.ceil(0.80 * len(abs_resid_tr))) - 1, len(abs_resid_tr) - 1)
        conf_margins[t_yr] = float(np.sort(abs_resid_tr)[max(0, k_idx)])

        r2_all      = r2_score(y_te, ens_gated) if len(y_te) > 1 and np.var(y_te) > 1e-16 else float('nan')
        rmse_f      = float(np.sqrt(mean_squared_error(y_te, ens_gated)))
        suppressed  = int((fire_prob < HURDLE_GATE_FOLD).sum())
        fire_recall = float(np.sum((ens_gated > 0) & (y_te > 0)) / max(np.sum(y_te > 0), 1))
        print(f"  Fold {t_yr} (fire-season n={te_mask.sum()}) | R2={r2_all:.4f} | RMSE={rmse_f:.4e} | "
              f"Gate={HURDLE_GATE_FOLD:.2f} (sup={suppressed}) | Recall={fire_recall:.3f} | "
              f"Weights=[{w_ens[0]:.2f},{w_ens[1]:.2f},{w_ens[2]:.2f}] | Iso={calib_status}")

    HURDLE_GATE = float(np.median(_fold_gates)) if _fold_gates else 0.30
    best_f2     = float(np.mean(_fold_f2s)) if _fold_f2s else 0.0
    region_hurdle_gates[reg_name] = HURDLE_GATE
    region_hurdle_gate[reg_name]  = {'threshold': float(HURDLE_GATE), 'f2': float(best_f2)}

    region_hyperparams[reg_name] = {
        'cb_best_params': dict(cb_study.best_params),
        'cb_best_cv_r2': float(cb_study.best_value),
        'cb_best_cv_rmse': float(cb_study.best_value),
        'cb_n_trials': len(cb_study.trials),
        'xgb_best_params': dict(xgb_study.best_params),
        'xgb_best_cv_r2': float(xgb_study.best_value),
        'xgb_best_cv_rmse': float(xgb_study.best_value),
        'xgb_n_trials': len(xgb_study.trials),
        'et_params': dict(ET_PARAMS),
        'hurdle_gate_threshold': float(HURDLE_GATE),
        'hurdle_gate_f2': float(best_f2),
        'region_quantile_alpha': float(REGION_QUANTILE_ALPHA),
    }
    TUNED_HYPERPARAMS[reg_name] = {
        'CatBoost': {
            'optuna_best_params': {k: _json_safe(v) for k, v in cb_study.best_params.items()},
            'optuna_best_value_r2': float(cb_study.best_value),
            'optuna_best_value_mse': float(cb_study.best_value),
            'n_optuna_trials': int(nt),
            'final_params_used': {k: _json_safe(v) for k, v in CB_PARAMS.items()},
        },
        'XGBoost': {
            'optuna_best_params': {k: _json_safe(v) for k, v in xgb_study.best_params.items()},
            'optuna_best_value_r2': float(xgb_study.best_value),
            'optuna_best_value_mse': float(xgb_study.best_value),
            'n_optuna_trials': int(nt),
            'final_params_used': {k: _json_safe(v) for k, v in XGB_PARAMS.items()},
        },
        'ExtraTrees_fixed_not_tuned': {k: _json_safe(v) for k, v in ET_PARAMS.items()},
    }

    conformal_margins[reg_name]       = conf_margins
    non_fire_season_fpr[reg_name]     = {
        'false_positives': int(_nonfire_fp_cnt),
        'total_nonfire_negatives': int(_nonfire_tot_cnt),
        'fpr': float(_nonfire_fp_cnt / max(_nonfire_tot_cnt, 1)),
    }
    y_pred_final_ba[global_rows]      = y_pred_reg
    y_pred_ba_prehurdle[global_rows]  = y_pred_prehurdle_reg
    y_pred_raw_ens[global_rows]       = y_pred_raw_reg
    y_pred_ba_ridge[global_rows]      = y_pred_reg
    y_pred_ba_cb[global_rows]         = y_cb_reg
    y_pred_ba_xgb[global_rows]        = y_xgb_reg
    y_pred_ba_et[global_rows]         = y_et_reg
    fire_prob_all[global_rows]        = fire_prob_reg
    y_pred_hurdle_prob[global_rows]   = fire_prob_reg
    y_fire_prob[global_rows]          = fire_prob_reg
    y_true_fire_bin[global_rows]      = y_bin_reg
    y_pred_fire_bin[global_rows]      = pred_fire_bin_reg
    y_used_hurdle[global_rows]        = used_hurdle_reg
    y_hurdle_zeroed[global_rows]      = hurdle_zeroed_reg
    y_train_max_ba[global_rows]       = train_max_reg
    y_raw_ens_max_ba[global_rows]     = raw_ens_max_reg

    if imp_weight > 0:
        region_feature_importance[reg_name]  = (imp_accum_f / imp_weight, list(reg_feats))
        REGION_FEATURE_IMPORTANCE[reg_name]  = (imp_accum_f / imp_weight, list(reg_feats))
        all_region_feat_importance[reg_name] = (imp_accum_f / imp_weight, list(reg_feats))

    valid_g = (cal_yr_reg >= TEST_YEAR_START) & (~np.isnan(y_pred_reg))
    if valid_g.sum() > 1:
        print(f"[{reg_name}] FIRE-SEASON OVERALL R2: {r2_score(y_reg[valid_g], y_pred_reg[valid_g]):.4f} | "
              f"RMSE: {np.sqrt(mean_squared_error(y_reg[valid_g], y_pred_reg[valid_g])):.4e} | "
              f"Non-fire-season FPR: {non_fire_season_fpr[reg_name]['fpr']:.4f}")

for yr in test_years:
    ms, mn = 0.0, 0
    for r in region_defs.keys():
        if r in conformal_margins and yr in conformal_margins[r]:
            ms += conformal_margins[r][yr]; mn += 1
    conformal_margin_by_fold_domain[yr] = ms / max(1, mn)

region_hurdle_gates['Global'] = float(np.mean(list(region_hurdle_gates.values()))) if region_hurdle_gates else 0.30

# baseline models
_tr_global_mask = (YEAR_START + (time_idn // 12)) < TEST_YEAR_START
_mo_all = (time_idn % 12).astype(np.int32)
y_base_clim = np.zeros(n_rows, dtype=np.float32)
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid
    _pm_tr = _pm & _tr_global_mask
    for _m in range(12):
        _m_tr = _pm_tr & (_mo_all == _m)
        _cval = float(np.nanmean(y_ba_idn[_m_tr])) if _m_tr.sum() > 0 else 0.0
        y_base_clim[_pm & (_mo_all == _m)] = _cval

y_base_pers = y_lag1_global.copy().astype(np.float32)
y_base_prev = y_lag12_global.copy().astype(np.float32)

_X_ols_all = np.column_stack([np.ones(n_rows, dtype=np.float64), y_base_clim, y_base_pers, y_base_prev])
_ols_fit_m = _tr_global_mask & (time_idn >= 12)
try:
    _ols_coef, *_ = np.linalg.lstsq(_X_ols_all[_ols_fit_m], y_ba_idn[_ols_fit_m].astype(np.float64), rcond=None)
except Exception:
    _ols_coef = np.array([0.0, 1.0, 0.0, 0.0])
y_base_ols = np.maximum(0.0, (_X_ols_all @ _ols_coef)).astype(np.float32)

def _eval_baseline_dict(mask_eval):
    if mask_eval.sum() < 2:
        return {}
    yt = y_ba_idn[mask_eval]
    yp = y_pred_final_ba[mask_eval]
    model_rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    model_rho  = float(spearmanr(yt, yp)[0]) if np.std(yp) > 0 and np.std(yt) > 0 else 0.0
    out = {
        'model': {
            'rmse': model_rmse,
            'r2': float(r2_score(yt, yp)),
            'mae': float(mean_absolute_error(yt, yp)),
            'spearman_r': model_rho,
            'n': int(mask_eval.sum()),
        }
    }
    for bname, bpred in [
        ('climatology',  y_base_clim[mask_eval]),
        ('persistence',  y_base_pers[mask_eval]),
        ('previous_year', y_base_prev[mask_eval]),
        ('ols_combined', y_base_ols[mask_eval]),
    ]:
        b_rmse = float(np.sqrt(mean_squared_error(yt, bpred)))
        b_rho  = float(spearmanr(yt, bpred)[0]) if np.std(bpred) > 0 and np.std(yt) > 0 else 0.0
        out[bname] = {
            'rmse': b_rmse,
            'r2': float(r2_score(yt, bpred)),
            'mae': float(mean_absolute_error(yt, bpred)),
            'spearman_r': b_rho,
            'skill_score_vs_baseline': float(1.0 - model_rmse / max(b_rmse, 1e-12)),
        }
    return out

valid_all = ((YEAR_START + (time_idn // 12)) >= TEST_YEAR_START) & (~np.isnan(y_pred_final_ba))
baseline_evaluation = {'Domain': _eval_baseline_dict(valid_all), 'Per_Subregion': {}}
for _rn, _rdef in region_defs.items():
    _rm = _rdef[0] if isinstance(_rdef, tuple) else _rdef
    baseline_evaluation['Per_Subregion'][_rn] = _eval_baseline_dict(valid_all & _rm)

print(f"\n{'='*80}")
print(" BENCHMARK BASELINE COMPARISON (FIRE-SEASON TEST FOLDS 2012-2020)")
print(f"{'='*80}")
if 'model' in baseline_evaluation['Domain']:
    _dm = baseline_evaluation['Domain']
    print(f"  Model         : RMSE={_dm['model']['rmse']:.4e} | R2={_dm['model']['r2']:.4f} | Spearman r={_dm['model']['spearman_r']:.3f}")
    for _bn in ('climatology', 'persistence', 'previous_year', 'ols_combined'):
        _b = _dm[_bn]
        print(f"  {_bn:<14s}: RMSE={_b['rmse']:.4e} | Spearman r={_b['spearman_r']:.3f} | Skill Score={_b['skill_score_vs_baseline']:+.3f}")

test_mask0     = (time_idn >= (TEST_YEAR_START - YEAR_START) * 12)
valid_meta     = ~np.isnan(y_pred_final_ba)
calib_yrs      = np.array([YEAR_START + (t // 12) for t in time_idn])
pixel_mean_ba  = np.full(n_idn_pixels, np.nan, dtype=np.float32)
_calib_mask_pm = valid_meta & test_mask0
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid; _te = _pm & _calib_mask_pm
    if _te.sum() >= 1:
        pixel_mean_ba[_pid] = float(np.nanmean(y_ba_idn[_te]))
row_pm_ba = pixel_mean_ba[pixel_inv]
_iv = pixel_mean_ba[~np.isnan(pixel_mean_ba)]
if len(_iv) > 0:
    _iq = np.quantile(_iv, [0.0, .2, .4, .6, .8])
    iso_tiers = [('Q1', _iq[0], _iq[1]), ('Q2', _iq[1], _iq[2]), ('Q3', _iq[2], _iq[3]),
                 ('Q4', _iq[3], _iq[4]), ('Q5', _iq[4], 1e9)]
else:
    iso_tiers = [('Q1', 0, 1e9)]
pix_lats = unique_pixels[:, 0]; pix_lons = unique_pixels[:, 1]
feat_labels = ['Two-stage pre-test pruned regional features']


Export the Optuna-tuned hyperparameters.


In [ ]:
import os
import pickle
import pandas as pd

if 'TUNED_HYPERPARAMS' not in dir() or not TUNED_HYPERPARAMS:
    print("No TUNED_HYPERPARAMS available -- run the region training loop above first.")
else:
    _export_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

    _rows = []
    for _reg_name, _models in TUNED_HYPERPARAMS.items():
        for _model_name, _info in _models.items():
            if _model_name == 'ExtraTrees_fixed_not_tuned':
                for _pname, _pval in _info.items():
                    _rows.append(dict(region=_reg_name, model='ExtraTrees',
                                       optuna_tuned=False, param=_pname,
                                       value=_pval, source='fixed_constant'))
                continue
            _tuned_names = set(_info['optuna_best_params'].keys())
            for _pname, _pval in _info['optuna_best_params'].items():
                _rows.append(dict(region=_reg_name, model=_model_name,
                                   optuna_tuned=True, param=_pname,
                                   value=_pval, source='optuna_search'))
            for _pname, _pval in _info['final_params_used'].items():
                if _pname in _tuned_names:
                    continue
                _rows.append(dict(region=_reg_name, model=_model_name,
                                   optuna_tuned=False, param=_pname,
                                   value=_pval, source='fixed_constant'))
            _rows.append(dict(region=_reg_name, model=_model_name,
                               optuna_tuned=None, param='n_optuna_trials',
                               value=_info['n_optuna_trials'], source='search_budget'))
            _rows.append(dict(region=_reg_name, model=_model_name,
                               optuna_tuned=None, param='optuna_best_value_mse',
                               value=_info['optuna_best_value_mse'], source='search_result'))

    tuned_hyperparams_df = pd.DataFrame(_rows)

    _csv_path = os.path.join(_export_dir, 'bbur_forecaster_mainland_tuned_hyperparameters.csv')
    _pkl_path = os.path.join(_export_dir, 'bbur_forecaster_mainland_tuned_hyperparameters.pkl')

    tuned_hyperparams_df.to_csv(_csv_path, index=False)
    with open(_pkl_path, 'wb') as f:
        pickle.dump({
            'active_region': ACTIVE_REGION,
            'tuned_hyperparams_raw': TUNED_HYPERPARAMS,
            'tuned_hyperparams_df': tuned_hyperparams_df,
        }, f)

    print(f"[Hyperparameter export] CSV -> {_csv_path} ({len(tuned_hyperparams_df)} rows)")
    print(f"[Hyperparameter export] Pickle -> {_pkl_path}")


Map the model outputs to the variable names used by the downstream export cells.


In [ ]:
import numpy as np

if 'y_pred_final_ba' in dir():
    y_pred_ba_cb  = y_pred_final_ba.copy()
    y_pred_ba_xgb = y_pred_final_ba.copy()
    y_pred_ba_et  = y_pred_final_ba.copy()

y_pred_hurdle_prob = fire_prob_all.copy() if 'fire_prob_all' in dir() else np.full(len(y_pred_final_ba) if 'y_pred_final_ba' in dir() else 0, np.nan, dtype=np.float32)

if 'region_hurdle_gate' in dir():
    regional_hurdle_gates = {k: v['threshold'] for k, v in region_hurdle_gate.items()}
else:
    regional_hurdle_gates = {}

regional_hurdle_eval = {}

if 'region_feature_importance' in dir():
    all_region_feat_importance = {
        k: {'names': v[1], 'importance': v[0]}
        for k, v in region_feature_importance.items()
    }
else:
    all_region_feat_importance = {}

feat_labels = feature_names_ext if 'feature_names_ext' in dir() else []

if 'unique_pixels' in dir() and hasattr(unique_pixels, 'shape') and len(unique_pixels.shape) == 2 and unique_pixels.shape[1] >= 2:
    pix_lats = unique_pixels[:, 0]
    pix_lons = unique_pixels[:, 1]
else:
    pix_lats = np.array([])
    pix_lons = np.array([])


Compute global and regional evaluation metrics for the export.


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
test_mask0 = time_idn >= (TEST_YEAR_START - YEAR_START) * 12
test_eval = test_mask0 & ~np.isnan(y_pred_final_ba)
yt = y_ba_idn[test_eval]
yp = y_pred_final_ba[test_eval]
r2_ba = r2_score(yt, yp) if len(yt) > 1 else np.nan
rmse_ba = np.sqrt(mean_squared_error(yt, yp)) if len(yt) > 1 else np.nan
bias_ba = float(np.mean(yp - yt)) if len(yt) > 1 else np.nan
pearson = float(np.corrcoef(yt, yp)[0, 1]) if len(yt) > 1 else np.nan
_region_defs_eval = [('Myanmar_West', is_myanmar_west_pixel > 0.5), ('Northern_Lower_Mekong', is_northern_lower_mekong_pixel > 0.5), ('Southern_Lower_Mekong', is_southern_lower_mekong_pixel > 0.5)]
for reg_name, reg_flag in _region_defs_eval:
    reg_pixels = np.where(reg_flag)[0]
    if len(reg_pixels) == 0:
        continue
    reg_mask = np.isin(pixel_inv, reg_pixels)
    reg_eval_mask = test_eval & reg_mask
    ytr = y_ba_idn[reg_eval_mask]
    ypr = y_pred_final_ba[reg_eval_mask]
    if len(ytr) > 1:
        reg_r2 = r2_score(ytr, ypr)
        reg_rmse = np.sqrt(mean_squared_error(ytr, ypr))


Compute hurdle-classifier and regressor evaluation metrics for the export.


In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, matthews_corrcoef
_region_defs_eval = [('Myanmar_West', is_myanmar_west_pixel > 0.5), ('Northern_Lower_Mekong', is_northern_lower_mekong_pixel > 0.5), ('Southern_Lower_Mekong', is_southern_lower_mekong_pixel > 0.5)]
_region_hurdle_gates = globals().get('region_hurdle_gates', {})
clf_eval_mask = test_eval & y_used_hurdle & ~np.isnan(y_fire_prob) & ~np.isnan(y_true_fire_bin)

def classifier_metrics(mask, label):
    yt = y_true_fire_bin[mask].astype(int)
    yp_bin = y_pred_fire_bin[mask].astype(int)
    yp_prob = y_fire_prob[mask]
    if len(yt) < 2 or len(np.unique(yt)) < 2:
        print(f'--- {label} --- insufficient class variety to score (n={len(yt)})')
        return None
    acc = accuracy_score(yt, yp_bin)
    bal_acc = balanced_accuracy_score(yt, yp_bin)
    mcc = matthews_corrcoef(yt, yp_bin)
    prec, rec, f1, _ = precision_recall_fscore_support(yt, yp_bin, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(yt, yp_prob)
    except ValueError:
        auc = np.nan
    try:
        ap = average_precision_score(yt, yp_prob)
    except ValueError:
        ap = np.nan
    base_rate = float(yt.mean())
    cm = confusion_matrix(yt, yp_bin, labels=[0, 1])
    gate_str = f"{_region_hurdle_gates.get(label if label != 'Global (Mainland)' else 'Global', float('nan')):.2f}" if isinstance(_region_hurdle_gates.get(label if label != 'Global (Mainland)' else 'Global'), (int, float)) else 'unknown'
    print(f'--- {label} (n={len(yt):,}, operating threshold={gate_str}, base fire-rate={base_rate:.3f}) ---')
    print(f'  Accuracy : {acc:.4f}   Balanced Acc: {bal_acc:.4f}   MCC: {mcc:.4f}')
    print(f'  Precision: {prec:.4f}   Recall: {rec:.4f}   F1: {f1:.4f}')
    print(f'  ROC-AUC  : {auc:.4f}   PR-AUC   : {ap:.4f}   (no-skill PR-AUC = base fire-rate = {base_rate:.4f})')
    print(f'  Confusion matrix [[TN FP] [FN TP]]:\n{cm}')
    return dict(label=label, n=len(yt), accuracy=acc, balanced_accuracy=bal_acc, mcc=mcc, precision=prec, recall=rec, f1=f1, roc_auc=auc, pr_auc=ap, base_rate=base_rate, cm=cm)
clf_results = {'Global': classifier_metrics(clf_eval_mask, 'Global (Mainland)')}
for reg_name, reg_flag in _region_defs_eval:
    reg_row_mask = clf_eval_mask & np.isin(pixel_inv, np.where(reg_flag)[0])
    clf_results[reg_name] = classifier_metrics(reg_row_mask, reg_name)


Compute fire-month regressor metrics for the export.


In [ ]:
import scipy.stats

def regressor_metrics(mask, label):
    yt = y_ba_idn[mask]
    yp = y_pred_final_ba[mask]
    if mask.sum() < 2 or np.std(yt) == 0:
        print(f'--- {label} --- insufficient rows to score (n={int(mask.sum())})')
        return None
    r2 = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    bias = float(np.mean(yp - yt))
    spearman_rho, spearman_p = scipy.stats.spearmanr(yt, yp)
    print(f'--- {label} (n={int(mask.sum()):,}) ---')
    print(f'  R2: {r2:.4f}   RMSE: {rmse:.4e}   Bias: {bias:.4e} ({bias / (yt.mean() + 1e-12) * 100:+.1f}%)   Spearman rho: {spearman_rho:.4f}')
    return dict(label=label, n=int(mask.sum()), r2=r2, rmse=rmse, bias=bias, spearman_rho=float(spearman_rho), spearman_pvalue=float(spearman_p))
fire_mask_true = test_eval & (y_ba_idn > 1e-06)
reg_results = {'Global': regressor_metrics(fire_mask_true, 'Global (Mainland) -- fire months only')}
for reg_name, reg_flag in _region_defs_eval:
    reg_row_mask = fire_mask_true & np.isin(pixel_inv, np.where(reg_flag)[0])
    reg_results[reg_name] = regressor_metrics(reg_row_mask, f'{reg_name} -- fire months only')
_scopes_fn = [('Global', test_eval)] + [(reg_name, test_eval & np.isin(pixel_inv, np.where(reg_flag)[0])) for reg_name, reg_flag in _region_defs_eval]
for label, scope_mask in _scopes_fn:
    m = scope_mask & (y_ba_idn > 1e-06)
    if m.sum() > 0:
        fn_rate = float(np.mean(y_hurdle_zeroed[m]))


Save the results checkpoint.


In [ ]:
import numpy as np
import os

ckpt_path = '/kaggle/working/eval_checkpoint.npz' if os.path.exists('/kaggle/working') else 'eval_checkpoint.npz'
np.savez_compressed(ckpt_path,
                    y_pred_final_ba=y_pred_final_ba,
                    y_pred_ba_ridge=y_pred_ba_ridge,
                    y_train_max_ba=y_train_max_ba,
                    y_raw_ens_max_ba=y_raw_ens_max_ba,
                    y_hurdle_zeroed=y_hurdle_zeroed,
                    y_fire_prob=y_fire_prob,
                    y_true_fire_bin=y_true_fire_bin,
                    y_pred_fire_bin=y_pred_fire_bin,
                    y_used_hurdle=y_used_hurdle,
                    is_myanmar_west_pixel=is_myanmar_west_pixel,
                    is_northern_lower_mekong_pixel=is_northern_lower_mekong_pixel,
                    is_southern_lower_mekong_pixel=is_southern_lower_mekong_pixel,
                    y_ba_idn=y_ba_idn,
                    pixel_inv=pixel_inv,
                    pix_lats=pix_lats,
                    pix_lons=pix_lons,
                    test_eval=test_eval,
                    time_idn=time_idn,
                    test_years=test_years,
                    region_gate_names=np.array(list(region_hurdle_gates.keys())),
                    region_gate_values=np.array(list(region_hurdle_gates.values()), dtype=np.float32))


Compute per-pixel metric grids for the export.


In [ ]:
import numpy as np
import scipy.stats
from sklearn.metrics import r2_score, mean_squared_error
lat_u = np.unique(pix_lats)
lon_u = np.unique(pix_lons)
lat_idx = {l: i for i, l in enumerate(lat_u)}
lon_idx = {l: i for i, l in enumerate(lon_u)}
H, W = (len(lat_u), len(lon_u))
obs_grid = np.full((H, W), np.nan)
prd_grid = np.full((H, W), np.nan)
bias_grid = np.full((H, W), np.nan)
r2_grid = np.full((H, W), np.nan)
_MIN_ROWS_FOR_R2_METRIC = 4
_MIN_FIRE_FOR_R2_METRIC = 2
pval_grid = np.full((H, W), np.nan)
spearman_grid = np.full((H, W), np.nan)
spearman_pval_grid = np.full((H, W), np.nan)
r2_clipped = []
n_r2_skipped_no_fire = 0
for pid in range(n_idn_pixels):
    pm = (pixel_inv == pid) & test_eval
    yt = y_ba_idn[pm]
    yp = y_pred_final_ba[pm]
    m_obs = np.mean(yt)
    m_prd = np.mean(yp)
    if np.std(yt) > 0 and np.std(yp) > 0:
        if int((yt > 0).sum()) < 2:
            n_r2_skipped_no_fire += 1
            continue
        r2 = r2_score(yt, yp)
        _, p = scipy.stats.pearsonr(yt, yp)
        rho, p_rho = scipy.stats.spearmanr(yt, yp)
    else:
        r2 = np.nan
        p = np.nan
        rho = np.nan
        p_rho = np.nan
    if not np.isnan(r2) and r2 >= -1.0:
        r2_clipped.append(r2)
    r = lat_idx[pix_lats[pid]]
    c = lon_idx[pix_lons[pid]]
    obs_grid[r, c] = m_obs
    prd_grid[r, c] = m_prd
    bias_grid[r, c] = m_prd - m_obs
    r2_grid[r, c] = r2
    pval_grid[r, c] = p
    spearman_grid[r, c] = rho
    spearman_pval_grid[r, c] = p_rho
r2_clipped = np.array(r2_clipped)


Compute domain-total monthly series and conformal intervals for the export.


In [ ]:
conformal_margins['Global'] = {int(yr): float(conformal_margin_by_fold_domain.get(int(yr), np.nan)) for yr in test_years}
_g_vals = [v for v in conformal_margins['Global'].values() if np.isfinite(v)]
conformal_margins['Global']['overall'] = float(np.mean(_g_vals)) if _g_vals else float('nan')
for _rn_c, _rm_c in list(conformal_margins.items()):
    if _rn_c == 'Global':
        continue
    _vals_c = [v for k, v in _rm_c.items() if k != 'overall' and np.isfinite(v)]
    _rm_c['overall'] = float(np.mean(_vals_c)) if _vals_c else float('nan')


Compute per-pixel hurdle-classifier and RMSE grids for the export.


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
MIN_CLF_SAMPLES_PER_PIXEL = 6
rmse_grid = np.full((H, W), np.nan)
prec_grid = np.full((H, W), np.nan)
rec_grid = np.full((H, W), np.nan)
f1_grid = np.full((H, W), np.nan)
n_reg_grid = np.full((H, W), np.nan)
n_clf_grid = np.full((H, W), np.nan)
for pid in range(n_idn_pixels):
    r = lat_idx[pix_lats[pid]]
    c = lon_idx[pix_lons[pid]]
    pm_reg = (pixel_inv == pid) & test_eval
    n_reg_grid[r, c] = int(pm_reg.sum())
    if pm_reg.sum() > 1:
        rmse_grid[r, c] = np.sqrt(mean_squared_error(y_ba_idn[pm_reg], y_pred_final_ba[pm_reg]))
    pm_clf = (pixel_inv == pid) & clf_eval_mask
    n_clf = int(pm_clf.sum())
    n_clf_grid[r, c] = n_clf
    if n_clf >= MIN_CLF_SAMPLES_PER_PIXEL:
        yt_pix = y_true_fire_bin[pm_clf].astype(int)
        yp_pix = y_pred_fire_bin[pm_clf].astype(int)
        p, r_, f, _ = precision_recall_fscore_support(yt_pix, yp_pix, average='binary', zero_division=0)
        prec_grid[r, c] = p
        rec_grid[r, c] = r_
        f1_grid[r, c] = f


Export the comprehensive final results (predictions, metrics and grids) to a pickle.


In [ ]:
import pickle, os, datetime

_export_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
_export_path = os.path.join(_export_dir, 'bbur_forecaster_mainland_final_comprehensive.pkl')

_pixel_region_label = np.full(n_idn_pixels, '', dtype=object)
for reg_name, reg_flag in _region_defs_eval:
    _pixel_region_label[reg_flag] = reg_name

_pix_r = np.array([lat_idx[la] for la in pix_lats])
_pix_c = np.array([lon_idx[lo] for lo in pix_lons])

pixel_table = {
    'pixel_id':         np.arange(n_idn_pixels),
    'lat':              pix_lats,
    'lon':              pix_lons,
    'region_label':     _pixel_region_label,
    'obs_mean_ba':      obs_grid[_pix_r, _pix_c],
    'pred_mean_ba':     prd_grid[_pix_r, _pix_c],
    'bias':             bias_grid[_pix_r, _pix_c],
    'r2':               r2_grid[_pix_r, _pix_c],
    'pearson_pvalue':   pval_grid[_pix_r, _pix_c],
    'spearman_r':       spearman_grid[_pix_r, _pix_c],
    'spearman_pvalue':  spearman_pval_grid[_pix_r, _pix_c],
    'rmse':             rmse_grid[_pix_r, _pix_c],
    'n_reg_samples':    n_reg_grid[_pix_r, _pix_c],
    'hurdle_precision': prec_grid[_pix_r, _pix_c],
    'hurdle_recall':    rec_grid[_pix_r, _pix_c],
    'hurdle_f1':        f1_grid[_pix_r, _pix_c],
    'n_clf_samples':    n_clf_grid[_pix_r, _pix_c],
}

test_months = np.unique(time_idn)
timeseries = {
    'months_since_2003': test_months,
    'calendar_year':     (YEAR_START + (test_months // 12)),
}
_scope_iter = [('Global', np.ones(len(y_ba_idn), dtype=bool))] + \
    [(reg_name, np.isin(pixel_inv, np.where(reg_flag)[0])) for reg_name, reg_flag in _region_defs_eval]
for scope_name, scope_mask in _scope_iter:
    obs_s, prd_s = [], []
    for tm in test_months:
        pm = (time_idn == tm) & scope_mask
        obs_s.append(float(np.nansum(y_ba_idn[pm])))
        prd_s.append(float(np.nansum(y_pred_final_ba[pm])))
    timeseries[scope_name] = {'observed_total': np.array(obs_s), 'predicted_total': np.array(prd_s)}

final_export = {
    'baseline_evaluation':       baseline_evaluation,
    'non_fire_season_fpr':       non_fire_season_fpr,
    'conformal_margins_by_region_fold': conformal_margins,
    'conformal_margin_by_fold_domain': conformal_margin_by_fold_domain,
    'region':           ACTIVE_REGION,
    'sub_regions':      [r for r, _ in _region_defs_eval],
    'lead_time':        'lead1',
    'test_years':       test_years,
    'test_year_start':  TEST_YEAR_START,
    'year_end':         YEAR_END,
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),

    'lat_u': lat_u, 'lon_u': lon_u,
    'grids': {
        'obs_mean_ba': obs_grid, 'pred_mean_ba': prd_grid, 'bias': bias_grid,
        'r2': r2_grid, 'pearson_pvalue': pval_grid,
        'spearman_r': spearman_grid, 'spearman_pvalue': spearman_pval_grid,
        'rmse': rmse_grid,
        'hurdle_precision': prec_grid, 'hurdle_recall': rec_grid, 'hurdle_f1': f1_grid,
        'n_reg_samples': n_reg_grid, 'n_clf_samples': n_clf_grid,
    },

    'pixel_table': pixel_table,

    'timeseries': timeseries,
    'conformal_margins': conformal_margins,

    'hurdle_classifier_metrics': clf_results,
    'regressor_metrics':         reg_results,

    'feature_importance': REGION_FEATURE_IMPORTANCE if 'REGION_FEATURE_IMPORTANCE' in dir() else {},
}

with open(_export_path, 'wb') as f:
    pickle.dump(final_export, f)

print(f'[Final Export] Saved comprehensive pickle -> {_export_path}')
print(f'  Top-level keys: {list(final_export.keys())}')
print(f'  Pixel table rows: {n_idn_pixels:,}  |  Test years: {test_years}')


Define the manuscript predictor categories.


In [ ]:
MANUSCRIPT_CATEGORIES = [
    'Static land-surface / fuel-load state',
    'Climate teleconnection & compound-drought indices',
    'Meteorological reanalysis & derived moisture amplifiers (ERA5)',
    'Fire-history predictors (GFED4s)',
    'Vegetation fuel-moisture state',
    'Other (spatial / calendar / region-specific)',
]

def categorize_feature(name: str) -> str:
    f = name.lower()

    if (f.startswith('defo_ba') or f.startswith('crop_ba')
            or f in ('years_since_first_defo', 'years_since_last_fire',
                     'fire_freq_train', 'fire_freq', 'fire_frequency',
                     'defo_recent_pressure_3_6mo', 'crop_recent_pressure_1_3mo',
                     'ba_prev_year', 'defo_x_dryseason', 'defo_x_dryseason_lag1',
                     'y_lag1', 'y_lag3')
            or 'cumulative_ba' in f
            or f.startswith('burned_area_pred_lag')):
        return MANUSCRIPT_CATEGORIES[3]

    if f.startswith('ndvi') or f.startswith('evi'):
        return MANUSCRIPT_CATEGORIES[4]

    if (f in ('soc', 'soc_z', 'soc_zscore', 'soc_stock', 'static_soc',
              'soc_domain_zscore', 'soc_domain', 'peat_mask', 'peat_mask_continuous')
            or 'soil_organic_carbon' in f):
        return MANUSCRIPT_CATEGORIES[0]

    if (f.startswith('oni') or f.startswith('dmi') or f.startswith('iod')
            or f.startswith('enso')
            or f in ('oni_x_dmi', 'oni_x_dmi_lag1', 'iod_x_oni', 'iod_x_oni_lag1',
                     'png_drought_x_oni', 'png_drought_x_oni_lag1', 'png_iod_drought', 'pm_drought_x_oni', 'pm_drought_x_oni_lag1',
                     'drought_oni_soft_interact', 'drought_oni_soft_interact_lag1',
                     'is_compound_drought', 'is_compound_drought_lag1', 'strong_enso')
            or 'kbdi' in f or f.startswith('drought_stress')):
        return MANUSCRIPT_CATEGORIES[1]

    _met_markers = (
        'era5_', 'dewpoint', 'd2m', 'soil_temp', 'stl', 'swvl',
        't2m', 'u10', 'v10', 'u_850', 'v_850', 'z_850', 'w_850',
        'u_500', 'v_500', 'z_500', 'w_500', 'vpd', 'tp_', 'precip',
        'grace', 'tws', 'rainfall_deficit', 'monsoon_onset',
        'drought_breaker_flag', 'days_since_last_rain',
        'pm_coastal_x_vpd', 'pm_inland_x_swvl3deficit',
    )
    if any(m in f for m in _met_markers):
        return MANUSCRIPT_CATEGORIES[2]

    return MANUSCRIPT_CATEGORIES[5]

def build_category_map(feature_names):
    return {f: categorize_feature(f) for f in feature_names}

_all_known_feats = list(feature_names_idn_full) if 'feature_names_idn_full' in dir() else (
                   list(feature_names_ext) if 'feature_names_ext' in dir() else list(feature_names))
category_map = build_category_map(_all_known_feats)
print(f'Categorized {len(category_map)} features across {len(MANUSCRIPT_CATEGORIES)} manuscript categories.')


Define the grouped permutation-importance functions.


In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

def grouped_permutation_importance_by_fold(x_full, y_full, time_full, feature_names_full,
                                            reg_feat_names, row_mask, year_start, test_years,
                                            min_train=200, min_test=5, n_repeats=5,
                                            random_state=42, cb_proxy_params=None,
                                            n_repeats_feat=3, return_feature_records=False):
    if cb_proxy_params is None:
        cb_proxy_params = dict(iterations=300, depth=5, learning_rate=0.08,
                                loss_function='RMSE', random_seed=random_state,
                                verbose=0, task_type='CPU', thread_count=4)

    used_feats = [f for f in reg_feat_names if f in feature_names_full]
    missing = [f for f in reg_feat_names if f not in feature_names_full]
    if missing:
        print(f"    [grouped-perm-importance] {len(missing)} feature(s) not found in "
              f"the feature-name list, skipped: {missing[:5]}{'...' if len(missing) > 5 else ''}")
    feat_idx = [feature_names_full.index(f) for f in used_feats]

    row_mask = np.asarray(row_mask, dtype=bool)
    X_reg = x_full[row_mask][:, feat_idx].astype(np.float32)
    y_reg = y_full[row_mask].astype(np.float32)
    t_reg = time_full[row_mask]
    cal_yr_reg = year_start + (t_reg // 12)

    cat_of = {f: categorize_feature(f) for f in used_feats}
    cats_present = sorted(set(cat_of.values()),
                           key=lambda c: MANUSCRIPT_CATEGORIES.index(c))
    cols_by_cat = {c: [i for i, f in enumerate(used_feats) if cat_of[f] == c]
                   for c in cats_present}

    rng = np.random.default_rng(random_state)
    rng_feat = np.random.default_rng(random_state + 1)
    feat_records = []
    fold_records = []
    fold_r2 = {}

    for t_yr in test_years:
        tr_mask = cal_yr_reg < t_yr
        te_mask = (cal_yr_reg == t_yr) & (~np.isnan(y_pred_final_ba[row_mask]))
        if tr_mask.sum() < min_train or te_mask.sum() < min_test:
            continue

        X_tr, y_tr = X_reg[tr_mask], y_reg[tr_mask]
        X_te, y_te = X_reg[te_mask], y_reg[te_mask]

        proxy = CatBoostRegressor(**cb_proxy_params)
        proxy.fit(X_tr, y_tr)
        base_r2 = r2_score(y_te, proxy.predict(X_te)) if len(y_te) > 1 else np.nan
        fold_r2[t_yr] = base_r2

        for cat, cols in cols_by_cat.items():
            if not cols:
                continue
            drops = []
            for _ in range(n_repeats):
                X_perm = X_te.copy()
                perm_order = rng.permutation(len(X_perm))
                X_perm[:, cols] = X_perm[perm_order][:, cols]
                perm_r2 = r2_score(y_te, proxy.predict(X_perm)) if len(y_te) > 1 else np.nan
                drops.append(base_r2 - perm_r2)
            fold_records.append((t_yr, cat, float(np.mean(drops))))

        if return_feature_records:
            for _j, _fname in enumerate(used_feats):
                _fdrops = []
                for _ in range(n_repeats_feat):
                    X_perm = X_te.copy()
                    X_perm[:, _j] = X_perm[rng_feat.permutation(len(X_perm)), _j]
                    _fdrops.append(base_r2 - (r2_score(y_te, proxy.predict(X_perm))
                                              if len(y_te) > 1 else np.nan))
                feat_records.append((t_yr, _fname, cat_of[_fname], float(np.mean(_fdrops))))

    if return_feature_records:
        return fold_records, cols_by_cat, fold_r2, feat_records
    return fold_records, cols_by_cat, fold_r2

def summarize_grouped_importance(fold_records, cols_by_cat):
    df = pd.DataFrame(fold_records, columns=['test_year', 'category', 'delta_r2'])
    rows = []
    for cat, cols in cols_by_cat.items():
        sub = df.loc[df.category == cat, 'delta_r2'].values
        if len(sub) == 0:
            continue
        rows.append(dict(
            category=cat,
            n_features=len(cols),
            n_folds=len(sub),
            mean_importance=float(np.mean(sub)),
            std_across_folds=float(np.std(sub)),
            per_feature_normalized=float(np.mean(sub)) / max(len(cols), 1),
        ))
    summary = pd.DataFrame(rows)
    if len(summary) == 0:
        return summary
    total_pos = summary['mean_importance'].clip(lower=0).sum()
    summary['pct_share'] = (summary['mean_importance'].clip(lower=0) / total_pos * 100
                             if total_pos > 0 else 0.0)
    order = {c: i for i, c in enumerate(MANUSCRIPT_CATEGORIES)}
    summary['_order'] = summary['category'].map(order)
    return summary.sort_values('_order').drop(columns='_order').reset_index(drop=True)

def summarize_top_features(feat_records, grouped_summary=None, top_n=5):
    df = pd.DataFrame(feat_records, columns=['test_year', 'feature', 'category', 'delta_r2'])
    if len(df) == 0:
        return pd.DataFrame()
    g = (df.groupby(['feature', 'category'])['delta_r2']
           .agg(mean_importance='mean', std_across_folds='std', n_folds='count')
           .reset_index())
    g['std_across_folds'] = g['std_across_folds'].fillna(0.0)
    g = g.sort_values('mean_importance', ascending=False).head(top_n).reset_index(drop=True)
    g.insert(0, 'rank', np.arange(1, len(g) + 1))
    if grouped_summary is not None and len(grouped_summary) > 0:
        g['category_pct_share'] = g['category'].map(
            dict(zip(grouped_summary['category'], grouped_summary['pct_share'])))
    else:
        g['category_pct_share'] = np.nan
    return g

def print_top_features(top_df, header, top_n=5):
    if top_df is None or len(top_df) == 0:
        print(f"  [{header}] No feature-level importance available.")
        return
    print(f"\n  >>> FINAL TOP {min(top_n, len(top_df))} FEATURES -- {header} "
          f"(single-feature permutation, mean dR2 across walk-forward folds)")
    print(f"  {'#':>2s}  {'feature':<34s} {'category':<62s} {'mean_dR2':>10s} "
          f"{'std_folds':>10s} {'cat_%share':>10s}")
    for _, r in top_df.iterrows():
        _sh = f"{r['category_pct_share']:>9.1f}%" if pd.notna(r['category_pct_share']) else f"{'n/a':>10s}"
        print(f"  {int(r['rank']):>2d}  {r['feature']:<34s} {r['category']:<62s} "
              f"{r['mean_importance']:>10.5f} {r['std_across_folds']:>10.5f} {_sh}")

def summarize_overall_top_features(region_feature_records, top_n=5):
    rows = []
    for _reg, _recs in region_feature_records.items():
        for (_yr, _f, _cat, _d) in _recs:
            rows.append((_reg, _yr, _f, _cat, _d))
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=['region', 'test_year', 'feature', 'category', 'delta_r2'])
    per_reg = df.groupby(['region', 'feature', 'category'])['delta_r2'].mean().reset_index()
    ov = (per_reg.groupby(['feature', 'category'])
                 .agg(mean_importance=('delta_r2', 'mean'), n_regions=('region', 'nunique'))
                 .reset_index()
                 .sort_values('mean_importance', ascending=False)
                 .head(top_n).reset_index(drop=True))
    ov.insert(0, 'rank', np.arange(1, len(ov) + 1))
    return ov

def print_overall_top_features(ov_df, n_regions_total, top_n=5):
    if ov_df is None or len(ov_df) == 0:
        return
    print(f"\n  >>> FINAL TOP {min(top_n, len(ov_df))} FEATURES -- ALL REGIONS COMBINED "
          f"(mean over the regions where the feature is a model input)")
    print(f"  {'#':>2s}  {'feature':<34s} {'category':<62s} {'mean_dR2':>10s} {'regions':>8s}")
    for _, r in ov_df.iterrows():
        print(f"  {int(r['rank']):>2d}  {r['feature']:<34s} {r['category']:<62s} "
              f"{r['mean_importance']:>10.5f} {int(r['n_regions']):>4d}/{n_regions_total}")


Run grouped permutation importance and report the top-5 predictors for each region.


In [ ]:
if 'REGION_FEATURE_IMPORTANCE' not in dir() or not REGION_FEATURE_IMPORTANCE:
    print("No REGION_FEATURE_IMPORTANCE available -- run the training cell above first.")
else:
    region_grouped_summaries = {}
    region_fold_records = {}
    region_feature_records = {}
    region_top5_features = {}

    for reg_name, (row_mask, _pix_mask) in region_defs.items():
        if reg_name not in REGION_FEATURE_IMPORTANCE:
            print(f"[{reg_name}] No stored feature list -- skipping.")
            continue
        _reg_feat_names = list(REGION_FEATURE_IMPORTANCE[reg_name].keys())
        print(f"\n[{reg_name}] Grouped permutation importance "
              f"({len(_reg_feat_names)} features, walk-forward folds = {test_years})...")

        _fold_records, _cols_by_cat, _fold_r2, _feat_records = grouped_permutation_importance_by_fold(
            x_full=x_idn, y_full=y_ba_idn, time_full=time_idn,
            feature_names_full=feature_names_idn_full,
            reg_feat_names=_reg_feat_names, row_mask=row_mask,
            year_start=YEAR_START, test_years=test_years,
            return_feature_records=True,
        )
        region_fold_records[reg_name] = _fold_records
        region_feature_records[reg_name] = _feat_records
        _summary = summarize_grouped_importance(_fold_records, _cols_by_cat)
        region_grouped_summaries[reg_name] = _summary

        print(f"  Per-fold proxy R2: " +
              ", ".join(f"{yr}={r2:.3f}" for yr, r2 in _fold_r2.items()))
        if len(_summary) == 0:
            print("  No folds had enough data to score -- skipping this region.")
            continue
        print(f"  {'Category':<62s} {'n_feat':>6s} {'mean_dR2':>10s} {'std_folds':>10s} "
              f"{'per_feat':>10s} {'%share':>8s}")
        for _, r in _summary.iterrows():
            print(f"  {r['category']:<62s} {r['n_features']:>6d} {r['mean_importance']:>10.5f} "
                  f"{r['std_across_folds']:>10.5f} {r['per_feature_normalized']:>10.5f} "
                  f"{r['pct_share']:>7.1f}%")

        _top5 = summarize_top_features(_feat_records, _summary, top_n=5)
        region_top5_features[reg_name] = _top5
        print_top_features(_top5, reg_name, top_n=5)

    region_overall_top5 = summarize_overall_top_features(region_feature_records, top_n=5)
    print_overall_top_features(region_overall_top5, n_regions_total=len(region_feature_records), top_n=5)


Export the grouped feature-importance results.


In [ ]:
import os
import pickle
import pandas as pd

if 'region_grouped_summaries' not in dir() or not region_grouped_summaries:
    print("No region_grouped_summaries available -- run the grouped-importance "
          "cell above first.")
else:
    _export_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

    _summary_rows = []
    for _reg_name, _summary in region_grouped_summaries.items():
        if _summary is None or len(_summary) == 0:
            continue
        _s = _summary.copy()
        _s.insert(0, 'region', _reg_name)
        _summary_rows.append(_s)
    grouped_importance_summary_df = (
        pd.concat(_summary_rows, ignore_index=True) if _summary_rows else pd.DataFrame()
    )

    _fold_rows = []
    for _reg_name, _fold_records in region_fold_records.items():
        for _test_year, _category, _delta_r2 in _fold_records:
            _fold_rows.append(dict(region=_reg_name, test_year=_test_year,
                                    category=_category, delta_r2=_delta_r2))
    grouped_importance_folds_df = pd.DataFrame(_fold_rows)

    _summary_path = os.path.join(
        _export_dir, 'bbur_forecaster_mainland_grouped_feature_importance_summary.csv')
    _folds_path = os.path.join(
        _export_dir, 'bbur_forecaster_mainland_grouped_feature_importance_folds.csv')
    _pickle_path = os.path.join(
        _export_dir, 'bbur_forecaster_mainland_grouped_feature_importance.pkl')

    grouped_importance_summary_df.to_csv(_summary_path, index=False)
    grouped_importance_folds_df.to_csv(_folds_path, index=False)
    with open(_pickle_path, 'wb') as f:
        pickle.dump({
            'active_region': ACTIVE_REGION,
            'manuscript_categories': MANUSCRIPT_CATEGORIES,
            'summary': grouped_importance_summary_df,
            'fold_records': grouped_importance_folds_df,
            'region_fold_records_raw': region_fold_records,
        }, f)

    print(f"[Feature-importance export] Summary CSV -> {_summary_path} "
          f"({len(grouped_importance_summary_df)} rows)")
    print(f"[Feature-importance export] Fold-level CSV -> {_folds_path} "
          f"({len(grouped_importance_folds_df)} rows)")
    print(f"[Feature-importance export] Combined pickle -> {_pickle_path}")
